In [1]:
!pip install implicit --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 938.7 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# ============================================
# CELL 1 — IMPORTS + CONSTANTS
# ============================================
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# Core column names
COL_USER   = "reviewerID"
COL_ITEM   = "asin"
COL_RATING = "overall"
COL_DATE   = "review_date"

CATEGORY = "electronics"

# ---- INPUT ARTIFACTS ----
# đổi path nếu dataset Kaggle của bạn đang mount tên khác
STAGE0_DIR = Path("/kaggle/input/datasets/datdong123/home-and-kitchen-stage-0/data/stage0_artifacts")
STAGE2_DIR = Path("/kaggle/input/datasets/datdong123/home-and-kitchen-stage-2/data/stage2_artifacts")

# ---- OUTPUT DIR ----
STAGE3_DIR = Path("/kaggle/working/data/stage3_artifacts")
STAGE3_DIR.mkdir(parents=True, exist_ok=True)

print("Stage0 input :", STAGE0_DIR)
print("Stage2 input :", STAGE2_DIR)
print("Stage3 output:", STAGE3_DIR)

Stage0 input : /kaggle/input/datasets/datdong123/home-and-kitchen-stage-0/data/stage0_artifacts
Stage2 input : /kaggle/input/datasets/datdong123/home-and-kitchen-stage-2/data/stage2_artifacts
Stage3 output: /kaggle/working/data/stage3_artifacts


In [3]:
# ============================================
# CELL 2 — LOAD STAGE 0 / STAGE 2 ARTIFACTS
# ============================================

# ---------- Stage 0 ----------
with open(STAGE0_DIR / "split_meta.json", "r") as f:
    split_meta = json.load(f)

with open(STAGE0_DIR / "baseline_scores.json", "r") as f:
    baseline_scores = json.load(f)

train   = pd.read_parquet(STAGE0_DIR / "train.parquet").copy()
test    = pd.read_parquet(STAGE0_DIR / "test.parquet").copy()
df_core = pd.read_parquet(STAGE0_DIR / "df_core.parquet").copy()

user_means_df = pd.read_parquet(STAGE0_DIR / "user_means.parquet")
item_means_df = pd.read_parquet(STAGE0_DIR / "item_means.parquet")

# ---------- Stage 2 ----------
with open(STAGE2_DIR / "encodings.pkl", "rb") as f:
    enc = pickle.load(f)

user_enc_s2    = enc["user_enc_s2"]
item_enc_s2_cf = enc["item_enc_s2_cf"]
item_enc_s2    = enc["item_enc_s2"]

user_means_vec = enc["user_means_vec"]
item_means_vec = enc["item_means_vec"]
GLOBAL_MEAN    = float(enc["global_mean"])

M_norm        = sp.load_npz(STAGE2_DIR / "M_norm.npz")
C_implicit    = sp.load_npz(STAGE2_DIR / "C_implicit.npz")
item_profiles = np.load(STAGE2_DIR / "item_profiles.npy")
user_profiles = np.load(STAGE2_DIR / "user_profiles.npy")

# ---------- Basic cleanup ----------
if COL_DATE in train.columns:
    train[COL_DATE] = pd.to_datetime(train[COL_DATE], errors="coerce")
if COL_DATE in test.columns:
    test[COL_DATE] = pd.to_datetime(test[COL_DATE], errors="coerce")
if COL_DATE in df_core.columns:
    df_core[COL_DATE] = pd.to_datetime(df_core[COL_DATE], errors="coerce")

# ---------- Mean maps ----------
# hỗ trợ cả trường hợp reviewerID/asin đang là cột thường hoặc index
if COL_USER in user_means_df.columns:
    user_mean_map = dict(zip(user_means_df[COL_USER], user_means_df["user_mean"]))
else:
    user_mean_map = user_means_df["user_mean"].to_dict()

if COL_ITEM in item_means_df.columns:
    item_mean_map = dict(zip(item_means_df[COL_ITEM], item_means_df["item_mean"]))
else:
    item_mean_map = item_means_df["item_mean"].to_dict()

print("=" * 70)
print("Loaded Stage 0 / Stage 2 artifacts")
print("=" * 70)

print(f"train shape        : {train.shape}")
print(f"test shape         : {test.shape}")
print(f"df_core shape      : {df_core.shape}")

print(f"GLOBAL_MEAN        : {GLOBAL_MEAN:.6f}")
print(f"Baseline warm RMSE : {baseline_scores['warm_rmse']:.6f}")
print(f"Baseline warm MAE  : {baseline_scores['warm_mae']:.6f}")

print(f"M_norm shape       : {M_norm.shape}")
print(f"C_implicit shape   : {C_implicit.shape}")
print(f"item_profiles      : {item_profiles.shape}")
print(f"user_profiles      : {user_profiles.shape}")

print(f"#user_enc_s2       : {len(user_enc_s2):,}")
print(f"#item_enc_s2       : {len(item_enc_s2):,}")
print(f"#item_enc_s2_cf    : {len(item_enc_s2_cf):,}")

Loaded Stage 0 / Stage 2 artifacts
train shape        : (4498249, 4)
test shape         : (1124563, 10)
df_core shape      : (5622812, 4)
GLOBAL_MEAN        : 4.340619
Baseline warm RMSE : 1.144186
Baseline warm MAE  : 0.855168
M_norm shape       : (620917, 169241)
C_implicit shape   : (620917, 169241)
item_profiles      : (169241, 128)
user_profiles      : (620917, 128)
#user_enc_s2       : 620,917
#item_enc_s2       : 169,241
#item_enc_s2_cf    : 169,241


In [4]:
# ============================================
# CELL 3 ? BUILD TEST_WARM + SANITY CHECKS + BASELINE REPLAY
# ============================================

assert len(train) == split_meta["n_train"], f"n_train mismatch: {len(train)} vs {split_meta['n_train']}"
assert len(test)  == split_meta["n_test"],  f"n_test mismatch: {len(test)} vs {split_meta['n_test']}"
assert M_norm.shape[0] == len(user_means_vec), "Mismatch: M_norm rows vs user_means_vec"
assert M_norm.shape[1] == len(item_means_vec), "Mismatch: M_norm cols vs item_means_vec"
assert item_profiles.shape[0] == len(item_enc_s2), "Mismatch: item_profiles rows vs item_enc_s2"
assert user_profiles.shape[0] == len(user_enc_s2), "Mismatch: user_profiles rows vs user_enc_s2"

train_users = set(train[COL_USER].unique())
train_items = set(train[COL_ITEM].unique())

test = test.copy()
test["user_known"] = test[COL_USER].isin(train_users)
test["item_known"] = test[COL_ITEM].isin(train_items)

mask_warm    = test["user_known"] & test["item_known"]
mask_cold_u  = ~test["user_known"] & test["item_known"]
mask_cold_i  =  test["user_known"] & ~test["item_known"]
mask_cold_ui = ~test["user_known"] & ~test["item_known"]

test_warm = test.loc[mask_warm].reset_index(drop=True)

def clip_rating(x):
    return np.clip(np.asarray(x, dtype=np.float32), 1.0, 5.0)

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return float(np.mean(np.abs(y_true - y_pred)))

def evaluate_predictions(y_true, y_pred, name="model"):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = clip_rating(y_pred)
    out = {"model": name, "n": int(len(y_true)), "rmse": rmse(y_true, y_pred), "mae": mae(y_true, y_pred), "per_star": {}}
    for s in [1, 2, 3, 4, 5]:
        mask = (y_true == s)
        if mask.sum() > 0:
            out["per_star"][f"star_{s}_rmse"] = rmse(y_true[mask], y_pred[mask])
            out["per_star"][f"star_{s}_mae"] = mae(y_true[mask], y_pred[mask])
            out["per_star"][f"star_{s}_n"] = int(mask.sum())
    return out

def print_eval(res):
    print("=" * 60)
    print("Model:", res["model"])
    print("N    :", res["n"])
    print("RMSE :", round(res["rmse"], 6))
    print("MAE  :", round(res["mae"], 6))
    print("-" * 60)
    for k, v in res["per_star"].items():
        print(f"{k:14s}: {v}")
    print("=" * 60)

def baseline_predict(df_subset, user_map, item_map, global_mean):
    u = df_subset[COL_USER].map(user_map).fillna(global_mean)
    i = df_subset[COL_ITEM].map(item_map).fillna(global_mean)
    return ((u + i) / 2.0).clip(1.0, 5.0).values.astype(np.float32)

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

def date_range_str(df_subset):
    if COL_DATE not in df_subset.columns or len(df_subset) == 0:
        return "n/a"
    valid_dates = df_subset[COL_DATE].dropna()
    if len(valid_dates) == 0:
        return "all NaT"
    return f"{valid_dates.min()} -> {valid_dates.max()}"

y_true_warm = test_warm[COL_RATING].values.astype(np.float32)
y_pred_base = baseline_predict(test_warm, user_mean_map, item_mean_map, GLOBAL_MEAN)
result_baseline = evaluate_predictions(y_true_warm, y_pred_base, "baseline_replay")
print_eval(result_baseline)

train_sorted = train.sort_values(COL_DATE, na_position="last").reset_index(drop=True)
split_idx_fit = int(len(train_sorted) * 0.90)
assert 0 < split_idx_fit < len(train_sorted), "Temporal split for train_fit/val_fit is invalid."
train_fit = train_sorted.iloc[:split_idx_fit].copy().reset_index(drop=True)
val_fit = train_sorted.iloc[split_idx_fit:].copy().reset_index(drop=True)

user_mean_map_fit = train_fit.groupby(COL_USER)[COL_RATING].mean().astype(np.float32).to_dict()
item_mean_map_fit = train_fit.groupby(COL_ITEM)[COL_RATING].mean().astype(np.float32).to_dict()
GLOBAL_MEAN_FIT = float(train_fit[COL_RATING].mean())

y_val_true_oof = val_fit[COL_RATING].astype(np.float32).values
baseline_val_preds_oof = baseline_predict(val_fit, user_mean_map_fit, item_mean_map_fit, GLOBAL_MEAN_FIT)
result_val_baseline_oof = evaluate_predictions(y_val_true_oof, baseline_val_preds_oof, "baseline_val_oof")

validation_index_cols = [COL_USER, COL_ITEM, COL_RATING]
if COL_DATE in val_fit.columns:
    validation_index_cols.append(COL_DATE)
if "user_group" in val_fit.columns:
    validation_index_cols.append("user_group")
validation_index_oof_df = val_fit.loc[:, validation_index_cols].copy()
validation_index_oof_df.insert(0, "oof_row_id", np.arange(len(validation_index_oof_df), dtype=np.int32))

print("Split summary")
print("-" * 60)
print("Expected split_date :", split_meta.get("split_date"))
print("Train               :", len(train))
print("Test                :", len(test))
print("Warm test           :", len(test_warm), f"({mask_warm.mean():.1%})")
print("Cold user           :", int(mask_cold_u.sum()))
print("Cold item           :", int(mask_cold_i.sum()))
print("Cold both           :", int(mask_cold_ui.sum()))
print("-" * 60)
print("Expected warm RMSE  :", baseline_scores["warm_rmse"])
print("Replay warm RMSE    :", result_baseline["rmse"])
print("Expected warm MAE   :", baseline_scores["warm_mae"])
print("Replay warm MAE     :", result_baseline["mae"])
print("-" * 60)
print("train_fit rows      :", len(train_fit))
print("val_fit rows        :", len(val_fit))
print("train_fit range     :", date_range_str(train_fit))
print("val_fit range       :", date_range_str(val_fit))
print("test_warm range     :", date_range_str(test_warm))
print("GLOBAL_MEAN_FIT     :", round(GLOBAL_MEAN_FIT, 6))
print("OOF baseline RMSE   :", round(result_val_baseline_oof["rmse"], 6))
print("OOF baseline MAE    :", round(result_val_baseline_oof["mae"], 6))

assert abs(result_baseline["rmse"] - baseline_scores["warm_rmse"]) < 0.02, "Baseline replay mismatch."
print("\nBaseline replay matched. test_warm stays final-eval only.")


Model: baseline_replay
N    : 935578
RMSE : 1.144186
MAE  : 0.855168
------------------------------------------------------------
star_1_rmse   : 3.215768337249756
star_1_mae    : 3.2064449787139893
star_1_n      : 60306
star_2_rmse   : 2.2479584217071533
star_2_mae    : 2.236931562423706
star_2_n      : 44048
star_3_rmse   : 1.2934361696243286
star_3_mae    : 1.2766953706741333
star_3_n      : 70544
star_4_rmse   : 0.3738032281398773
star_4_mae    : 0.3362816274166107
star_4_n      : 126808
star_5_rmse   : 0.6189844608306885
star_5_mae    : 0.5923428535461426
star_5_n      : 633872
Split summary
------------------------------------------------------------
Expected split_date : 2017-05-15 00:00:00
Train               : 4498249
Test                : 1124563
Warm test           : 935578 (83.2%)
Cold user           : 169780
Cold item           : 16539
Cold both           : 2666
------------------------------------------------------------
Expected warm RMSE  : 1.1441860397718033
Replay war

In [5]:
# ============================================
# CELL 4 ? MODEL C PREP
# ============================================

def l2_normalize_rows(X, eps=1e-12):
    X = np.asarray(X, dtype=np.float32)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.maximum(norms, eps)
    return X / norms
def build_content_artifacts(train_source_df, item_profiles_base, item_encoder_base):
    item_profiles_base = l2_normalize_rows(item_profiles_base)
    train_items = [item_id for item_id in train_source_df[COL_ITEM].drop_duplicates().tolist() if item_id in item_encoder_base]
    
    item_row_idx = np.asarray([item_encoder_base[item_id] for item_id in train_items], dtype=np.int32)
    item_profiles_local = item_profiles_base[item_row_idx].astype(np.float32)
    item_enc_local = {item_id: idx for idx, item_id in enumerate(train_items)}
    
    train_known = train_source_df[train_source_df[COL_ITEM].isin(item_enc_local)].copy().reset_index(drop=True)
    user_list = train_known[COL_USER].drop_duplicates().tolist()
    user_enc_local = {user_id: idx for idx, user_id in enumerate(user_list)}
    
    rows = train_known[COL_USER].map(user_enc_local).astype(np.int32).values
    cols = train_known[COL_ITEM].map(item_enc_local).astype(np.int32).values
    
    # --- FIX: Center the ratings ---
    # Instead of np.ones, use (rating - 3.5)
    # This allows negative similarities for disliked items
    vals = (train_known[COL_RATING].values - 3.5).astype(np.float32)
    
    user_item_csr = sp.csr_matrix((vals, (rows, cols)), shape=(len(user_list), len(item_enc_local)), dtype=np.float32)
    user_profiles_local = user_item_csr @ item_profiles_local
    
    # Use absolute counts for normalization to preserve the direction of the vector
    user_counts_local = np.asarray(np.abs(user_item_csr).sum(axis=1)).ravel().astype(np.float32)
    user_profiles_local = user_profiles_local / np.maximum(user_counts_local[:, None], 1.0)
    user_profiles_local = l2_normalize_rows(user_profiles_local)
    
    return {
        "user_enc": user_enc_local, 
        "item_enc": item_enc_local, 
        "user_profiles": user_profiles_local.astype(np.float32), 
        "item_profiles": item_profiles_local.astype(np.float32)
    }

def predict_content_weighted_with_fallback(
    df_subset,
    baseline_arr,
    content_artifacts,
    w_base,
    w_content,
    sim_threshold=0.0
):
    preds = baseline_arr.copy().astype(np.float32)
    sims = np.zeros(len(df_subset), dtype=np.float32)
    anchor = baseline_arr.copy().astype(np.float32)

    known_mask = df_subset[COL_USER].isin(content_artifacts["user_enc"]) & df_subset[COL_ITEM].isin(content_artifacts["item_enc"])
    effective_mask = np.zeros(len(df_subset), dtype=bool)

    if known_mask.any():
        known_df = df_subset.loc[known_mask].copy()
        u_idx = known_df[COL_USER].map(content_artifacts["user_enc"]).astype(np.int32).values
        i_idx = known_df[COL_ITEM].map(content_artifacts["item_enc"]).astype(np.int32).values

        U = content_artifacts["user_profiles"][u_idx]
        I = content_artifacts["item_profiles"][i_idx]

        sims_known = np.sum(U * I, axis=1).astype(np.float32)
        anchor_known = baseline_arr[known_mask.to_numpy()]
        use_mask = sims_known >= sim_threshold

        preds_known = anchor_known.copy()
        preds_known[use_mask] = clip_rating(
            w_base * anchor_known[use_mask] +
            w_content * sims_known[use_mask]
        )

        preds[known_mask.to_numpy()] = preds_known
        sims[known_mask.to_numpy()] = sims_known
        anchor[known_mask.to_numpy()] = anchor_known
        effective_mask[known_mask.to_numpy()] = use_mask

    return preds.astype(np.float32), sims.astype(np.float32), anchor.astype(np.float32), known_mask, effective_mask

item_profiles_norm = l2_normalize_rows(item_profiles)

item_profiles_norm = l2_normalize_rows(item_profiles)
print("item_profiles_norm shape:", item_profiles_norm.shape)
print("Ready to rebuild content profiles from train_fit and full train only.")


item_profiles_norm shape: (169241, 128)
Ready to rebuild content profiles from train_fit and full train only.


In [6]:
# ============================================
# CELL 5 ? MODEL C TUNING ON val_fit + SAVE OOF
# ============================================


print("=" * 80)
print("MODEL C — TUNE w_base + w_content + sim_threshold ON val_fit")
print("=" * 80)

content_fit_artifacts = build_content_artifacts(train_fit, item_profiles_norm, item_enc_s2)

# Combined grid to capture both standard performance and lower-rating improvements
w_base_grid = [0.8, 0.9, 1.0, 1.1]
w_content_grid = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
sim_threshold_grid = [0.0, 0.05, 0.1, 0.2]

results_c_grid = []

for wb in w_base_grid:
    for wc in w_content_grid:
        for th in sim_threshold_grid:

            # Unpack 5 variables from the prediction fallback function
            y_pred_tmp, sims_tmp, anchor_tmp, known_mask_tmp, eff_mask_tmp = predict_content_weighted_with_fallback(
                val_fit,
                baseline_val_preds_oof,
                content_fit_artifacts,
                w_base=wb,
                w_content=wc,
                sim_threshold=th
            )

            res_tmp = evaluate_predictions(
                y_val_true_oof,
                y_pred_tmp,
                name=f"model_c_val_wb_{wb}_wc_{wc}_th_{th}"
            )

            # Store comprehensive metrics for analysis
            results_c_grid.append({
                "w_base": float(wb),
                "w_content": float(wc),
                "sim_threshold": float(th),
                "rmse": float(res_tmp["rmse"]),
                "mae": float(res_tmp["mae"]),
                "n": int(res_tmp["n"]),
                "theoretical_coverage": float(known_mask_tmp.mean()), # User/item exist in profiles
                "effective_coverage": float(eff_mask_tmp.mean()),    # Weights actually applied
                "sim_mean": float(np.mean(sims_tmp)),
                "sim_std": float(np.std(sims_tmp))
            })

            # Verbose printing for real-time monitoring
            print(
                f"wb={wb:<4.2f} wc={wc:<4.2f} th={th:<4.2f} | "
                f"RMSE={res_tmp['rmse']:.6f} | "
                f"Eff. Coverage={float(eff_mask_tmp.mean()):.4f}"
            )

# Create DataFrame and apply the smart ranking
results_c_grid_df = pd.DataFrame(results_c_grid)

# Score balances RMSE with EFFECTIVE coverage to avoid over-filtering
results_c_grid_df["score"] = results_c_grid_df["rmse"] + 0.1 * (1 - results_c_grid_df["effective_coverage"])

# Sort by the custom score (prioritizing lower ratings and coverage), then RMSE, then MAE
results_c_grid_df = results_c_grid_df.sort_values(
    ["score", "rmse", "mae"],
    ascending=[True, True, True]
).reset_index(drop=True)

print("\n--- Top 5 Configurations ---")
display(results_c_grid_df.head(5))

# Extract the absolute best parameters
best_row = results_c_grid_df.iloc[0]
best_wb = float(best_row["w_base"])
best_wc = float(best_row["w_content"])
best_th = float(best_row["sim_threshold"])

print("\nBest params on val_fit:")
print(f"w_base        = {best_wb}")
print(f"w_content     = {best_wc}")
print(f"sim_threshold = {best_th}")
print(f"Effective Cov = {best_row['effective_coverage']:.4f}")

# Run the final OOF generation using the best parameters
model_c_val_preds_oof, model_c_val_sims_oof, model_c_val_anchor_oof, model_c_val_known_mask, model_c_val_eff_mask = predict_content_weighted_with_fallback(
    val_fit,
    baseline_val_preds_oof,
    content_fit_artifacts,
    w_base=best_wb,
    w_content=best_wc,
    sim_threshold=best_th
)

model_c_val_result_oof = evaluate_predictions(
    y_val_true_oof,
    model_c_val_preds_oof,
    name=f"model_c_val_oof_best"
)

print("\nFinal Model C OOF Evaluation:")
print_eval(model_c_val_result_oof)

# --- Save Artifacts to Kaggle Output ---
# Ensure the path exists within /kaggle/working/data
FINAL_OUTPUT_PATH = Path("/kaggle/working/data/stage3_artifacts")
FINAL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

np.save(FINAL_OUTPUT_PATH / "model_c_val_preds_oof.npy", model_c_val_preds_oof)
np.save(FINAL_OUTPUT_PATH / "model_c_val_sims_oof.npy", model_c_val_sims_oof)
results_c_grid_df.to_csv(FINAL_OUTPUT_PATH / "model_c_tuning.csv", index=False)

print(f"\n=== Model C Artifacts Saved to {FINAL_OUTPUT_PATH} ===")

MODEL C — TUNE w_base + w_content + sim_threshold ON val_fit
wb=0.80 wc=0.05 th=0.00 | RMSE=1.291072 | Eff. Coverage=0.7214
wb=0.80 wc=0.05 th=0.05 | RMSE=1.290256 | Eff. Coverage=0.7188
wb=0.80 wc=0.05 th=0.10 | RMSE=1.289555 | Eff. Coverage=0.7166
wb=0.80 wc=0.05 th=0.20 | RMSE=1.288455 | Eff. Coverage=0.7131
wb=0.80 wc=0.10 th=0.00 | RMSE=1.271005 | Eff. Coverage=0.7214
wb=0.80 wc=0.10 th=0.05 | RMSE=1.270178 | Eff. Coverage=0.7188
wb=0.80 wc=0.10 th=0.10 | RMSE=1.269472 | Eff. Coverage=0.7166
wb=0.80 wc=0.10 th=0.20 | RMSE=1.268373 | Eff. Coverage=0.7131
wb=0.80 wc=0.20 th=0.00 | RMSE=1.233612 | Eff. Coverage=0.7214
wb=0.80 wc=0.20 th=0.05 | RMSE=1.232764 | Eff. Coverage=0.7188
wb=0.80 wc=0.20 th=0.10 | RMSE=1.232048 | Eff. Coverage=0.7166
wb=0.80 wc=0.20 th=0.20 | RMSE=1.230953 | Eff. Coverage=0.7131
wb=0.80 wc=0.30 th=0.00 | RMSE=1.200153 | Eff. Coverage=0.7214
wb=0.80 wc=0.30 th=0.05 | RMSE=1.199286 | Eff. Coverage=0.7188
wb=0.80 wc=0.30 th=0.10 | RMSE=1.198563 | Eff. Coverage=0

,w_base,w_content,sim_threshold,rmse,mae,n,theoretical_coverage,effective_coverage,sim_mean,sim_std,score
0,0.9,0.40,0.10,1.094887,0.823112,449825,0.831032,0.716636,0.571073,0.599475,1.123223
1,0.9,0.40,0.05,1.095117,0.823478,449825,0.831032,0.718800,0.571073,0.599475,1.123237
2,0.9,0.40,0.00,1.095399,0.823938,449825,0.831032,0.721445,0.571073,0.599475,1.123255
3,0.9,0.40,0.20,1.094574,0.822570,449825,0.831032,0.713097,0.571073,0.599475,1.123264
4,1.0,0.05,0.00,1.096352,0.777602,449825,0.831032,0.721445,0.571073,0.599475,1.124208



Best params on val_fit:
w_base        = 0.9
w_content     = 0.4
sim_threshold = 0.1
Effective Cov = 0.7166

Final Model C OOF Evaluation:
Model: model_c_val_oof_best
N    : 449825
RMSE : 1.094887
MAE  : 0.823112
------------------------------------------------------------
star_1_rmse   : 3.0549373626708984
star_1_mae    : 3.011327028274536
star_1_n      : 24386
star_2_rmse   : 2.132822036743164
star_2_mae    : 2.0813775062561035
star_2_n      : 18751
star_3_rmse   : 1.233289122581482
star_3_mae    : 1.1703380346298218
star_3_n      : 32659
star_4_rmse   : 0.45014241337776184
star_4_mae    : 0.3809417188167572
star_4_n      : 62459
star_5_rmse   : 0.7255696058273315
star_5_mae    : 0.6283630728721619
star_5_n      : 311570

=== Model C Artifacts Saved to /kaggle/working/data/stage3_artifacts ===


In [7]:
# ============================================
# CELL 6 ? MODEL C REFIT ON FULL TRAIN + FINAL TEST_WARM SAVE
# ============================================



print("=" * 60)
print("MODEL C — REFIT USER PROFILES ON FULL TRAIN")
print("=" * 60)

# 1. Rebuild artifacts using the entire training set (train)
content_full_artifacts = build_content_artifacts(train, item_profiles_norm, item_enc_s2)

# 2. Predict using the BEST hyperparameters found in the tuning stage
# Note: Using 'predict_content_weighted_with_fallback' to match your new tuning logic
model_c_preds, model_c_sims, model_c_anchor, model_c_known_mask, model_c_eff_mask = predict_content_weighted_with_fallback(
    test_warm, 
    y_pred_base, 
    content_full_artifacts, 
    w_base=best_wb, 
    w_content=best_wc, 
    sim_threshold=best_th
)

assert len(model_c_preds) == len(test_warm), "Model C final predictions must align with test_warm."

# 3. Evaluation
result_c_best = evaluate_predictions(y_true_warm, model_c_preds, name=f"model_c_full_train_final")
result_base_c = evaluate_predictions(y_true_warm, y_pred_base, name="baseline_on_test_warm")

print("\n--- Baseline Results ---")
print_eval(result_base_c)
print("\n--- Model C Final Results ---")
print_eval(result_c_best)

# 4. Calculate Gains
rmse_gain_c = result_base_c["rmse"] - result_c_best["rmse"]
mae_gain_c  = result_base_c["mae"] - result_c_best["mae"]

model_c_summary = {
    "best_w_base": float(best_wb),
    "best_w_content": float(best_wc),
    "best_sim_threshold": float(best_th),
    "val_fit_rmse": float(model_c_val_result_oof["rmse"]),
    "test_warm_rmse": float(result_c_best["rmse"]),
    "rmse_gain_vs_baseline": float(rmse_gain_c),
    "test_warm_eff_coverage": float(model_c_eff_mask.mean()),
    "per_star": result_c_best["per_star"]
}

# 5. Save Artifacts to the correct Kaggle output directory
FINAL_STAGE3_DIR = Path("/kaggle/working/data/stage3_artifacts")
FINAL_STAGE3_DIR.mkdir(parents=True, exist_ok=True)

np.save(FINAL_STAGE3_DIR / "model_c_preds.npy", model_c_preds)
np.save(FINAL_STAGE3_DIR / "model_c_sims.npy", model_c_sims)
np.save(FINAL_STAGE3_DIR / "model_c_anchor.npy", model_c_anchor)

with open(FINAL_STAGE3_DIR / "model_c_result.json", "w") as f:
    json.dump(to_jsonable(result_c_best), f, indent=2)
    
with open(FINAL_STAGE3_DIR / "model_c_summary.json", "w") as f:
    json.dump(to_jsonable(model_c_summary), f, indent=2)

print(f"\n=== Model C Refit Complete & Saved to {FINAL_STAGE3_DIR} ===")

MODEL C — REFIT USER PROFILES ON FULL TRAIN

--- Baseline Results ---
Model: baseline_on_test_warm
N    : 935578
RMSE : 1.144186
MAE  : 0.855168
------------------------------------------------------------
star_1_rmse   : 3.215768337249756
star_1_mae    : 3.2064449787139893
star_1_n      : 60306
star_2_rmse   : 2.2479584217071533
star_2_mae    : 2.236931562423706
star_2_n      : 44048
star_3_rmse   : 1.2934361696243286
star_3_mae    : 1.2766953706741333
star_3_n      : 70544
star_4_rmse   : 0.3738032281398773
star_4_mae    : 0.3362816274166107
star_4_n      : 126808
star_5_rmse   : 0.6189844608306885
star_5_mae    : 0.5923428535461426
star_5_n      : 633872

--- Model C Final Results ---
Model: model_c_full_train_final
N    : 935578
RMSE : 1.147623
MAE  : 0.885929
------------------------------------------------------------
star_1_rmse   : 3.159782886505127
star_1_mae    : 3.1515564918518066
star_1_n      : 60306
star_2_rmse   : 2.189922571182251
star_2_mae    : 2.1799776554107666
star

Model C after pipeline cleanup and missing-mean fix achieved better MAE and improved low-star predictions, but it still underperformed the warm baseline in overall RMSE (1.2174 vs 1.2134). This suggests the content-based signal is useful as a fallback or ensemble component, but not strong enough as a standalone warm recommendation model. Therefore, the next priority should be Model A, which is more likely to surpass the baseline on the warm subset.

In [8]:
# ============================================
# CELL 7 ? MODEL A PREP (ITEM-ITEM CF)
# ============================================
from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd

print("=" * 70)
print("MODEL A ? ITEM-ITEM CF")
print("=" * 70)

MAX_K = 20

def build_item_item_cf_artifacts(train_source_df):
    user_list = train_source_df[COL_USER].drop_duplicates().tolist()
    item_list = train_source_df[COL_ITEM].drop_duplicates().tolist()
    user_enc_local = {user_id: idx for idx, user_id in enumerate(user_list)}
    item_enc_local = {item_id: idx for idx, item_id in enumerate(item_list)}
    rows = train_source_df[COL_USER].map(user_enc_local).astype(np.int32).values
    cols = train_source_df[COL_ITEM].map(item_enc_local).astype(np.int32).values
    vals = train_source_df[COL_RATING].astype(np.float32).values
    M_raw_local = sp.csr_matrix((vals, (rows, cols)), shape=(len(user_list), len(item_list)), dtype=np.float32)
    user_counts_local = np.diff(M_raw_local.indptr).astype(np.int32)
    user_sums_local = np.asarray(M_raw_local.sum(axis=1)).ravel().astype(np.float32)
    user_means_raw_local = user_sums_local / np.maximum(user_counts_local, 1)
    M_user_centered_local = M_raw_local.copy().astype(np.float32)
    row_idx_local = np.repeat(np.arange(M_user_centered_local.shape[0], dtype=np.int32), np.diff(M_user_centered_local.indptr))
    M_user_centered_local.data -= user_means_raw_local[row_idx_local]
    B_local = M_raw_local.copy().astype(np.float32)
    B_local.data = np.ones_like(B_local.data, dtype=np.float32)
    return {"user_enc": user_enc_local, "item_enc": item_enc_local, "M_raw": M_raw_local, "M_user_centered": M_user_centered_local, "B": B_local, "user_means_raw": user_means_raw_local}

def precompute_item_neighbors(user_centered_csr, max_k=20):
    X_items_local = user_centered_csr.T.tocsr()
    n_neighbors_local = min(max_k + 1, X_items_local.shape[0])
    assert n_neighbors_local >= 2, "Model A needs at least 2 items to compute neighbors."
    nn_model_local = NearestNeighbors(n_neighbors=n_neighbors_local, metric="cosine", algorithm="brute", n_jobs=-1)
    nn_model_local.fit(X_items_local)
    distances_local, indices_local = nn_model_local.kneighbors(X_items_local, return_distance=True)
    sims_raw_local = 1.0 - distances_local
    return indices_local[:, 1:].astype(np.int32), sims_raw_local[:, 1:].astype(np.float32)

model_a_fit_artifacts = build_item_item_cf_artifacts(train_fit)
user_enc_a_fit = model_a_fit_artifacts["user_enc"]
item_enc_a_fit = model_a_fit_artifacts["item_enc"]
user_means_raw_a_fit = model_a_fit_artifacts["user_means_raw"]

print("train_fit users:", len(user_enc_a_fit))
print("train_fit items:", len(item_enc_a_fit))
print("train_fit matrix shape:", model_a_fit_artifacts["M_raw"].shape)

MODEL A ? ITEM-ITEM CF
train_fit users: 601727
train_fit items: 166990
train_fit matrix shape: (601727, 166990)


In [9]:
# ============================================
# CELL 8 ? PRECOMPUTE ITEM NEIGHBORS FOR train_fit
# ============================================

item_neighbors_idx_a_fit, item_neighbors_sim_a_fit = precompute_item_neighbors(model_a_fit_artifacts["M_user_centered"], max_k=MAX_K)

print("item_neighbors_idx_a_fit shape:", item_neighbors_idx_a_fit.shape)
print("item_neighbors_sim_a_fit shape:", item_neighbors_sim_a_fit.shape)
print("Example item 0 neighbors   :", item_neighbors_idx_a_fit[0][:10])
print("Example item 0 sims        :", item_neighbors_sim_a_fit[0][:10])

item_neighbors_idx_a_fit shape: (166990, 20)
item_neighbors_sim_a_fit shape: (166990, 20)
Example item 0 neighbors   : [116441 133584 128320 115086 123748   2995  19893 124502 107472 145610]
Example item 0 sims        : [0.29356742 0.27940005 0.21360874 0.19517744 0.17397833 0.1484608
 0.13610613 0.13391757 0.10776311 0.10677111]


In [10]:
# ============================================
# CELL 9 ? CO-RATER COUNTS + PREDICT (FIXED)
# ============================================

import numpy as np
import time

def compute_neighbor_co_counts(B_csr, neighbor_idx_matrix):
    """
    Computes co-occurrence counts ONLY for the pre-selected neighbors.
    Memory footprint is O(Items * K) instead of O(Items^2).
    """
    print("Computing localized co-counts (lazy evaluation)...")
    t0 = time.time()
    
    n_items = neighbor_idx_matrix.shape[0]
    max_k = neighbor_idx_matrix.shape[1]
    
    # Store the results here
    co_counts = np.zeros((n_items, max_k), dtype=np.int32)
    
    # Convert B to CSC format once. 
    # CSC is highly optimized for slicing entire columns (items).
    B_csc = B_csr.tocsc()
    
    for item_idx in range(n_items):
        # 1. Get the sparse column representing the target item
        # Shape: (Users, 1)
        item_col = B_csc[:, item_idx]
        
        # 2. Get the neighbor indices for this item
        nbrs = neighbor_idx_matrix[item_idx]
        
        # 3. Extract a sub-matrix of ONLY the neighbors
        # Shape: (Users, K)
        nbr_cols = B_csc[:, nbrs]
        
        # 4. Perform localized sparse matrix multiplication: 
        # (1, Users) @ (Users, K) = (1, K)
        # This counts the exact common users natively in C++ via SciPy
        counts_for_this_item = (item_col.T @ nbr_cols).toarray().ravel()
        
        co_counts[item_idx] = counts_for_this_item
        
        # Optional progress tracker
        if item_idx > 0 and item_idx % 25000 == 0:
            print(f"  Processed {item_idx}/{n_items} items...")

    print(f"Lazy co-counts finished in {time.time() - t0:.1f}s")
    return co_counts

def predict_item_item_cf_fixed(df_subset, u_idx, i_idx, M_user_centered_csr, user_means_raw_vec, neighbor_idx_matrix, neighbor_sim_matrix, neighbor_common_matrix, user_mean_map_stage0, item_mean_map_stage0, global_mean, topk=50, min_common_users=2, min_neighbors_for_pred=3, shrinkage=10.0, min_sim_threshold=0.0):
    n = len(u_idx)
    preds = np.empty(n, dtype=np.float32)
    used_neighbor_counts = np.zeros(n, dtype=np.int32)
    fallback_flags = np.zeros(n, dtype=np.int8)
    raw_user_fallback = df_subset[COL_USER].map(user_mean_map_stage0).fillna(global_mean).astype(np.float32).values
    raw_item_fallback = df_subset[COL_ITEM].map(item_mean_map_stage0).fillna(global_mean).astype(np.float32).values
    baseline_fallback = 0.5 * (raw_user_fallback + raw_item_fallback)
    for t in range(n):
        u = u_idx[t]
        i = i_idx[t]
        row = M_user_centered_csr.getrow(u)
        rated_items = row.indices
        rated_vals = row.data
        if len(rated_items) == 0:
            preds[t] = baseline_fallback[t]
            fallback_flags[t] = 1
            continue
        nbr_idx = neighbor_idx_matrix[i, :topk]
        nbr_sim = neighbor_sim_matrix[i, :topk]
        nbr_cnt = neighbor_common_matrix[i, :topk]
        reliable_mask = (nbr_cnt >= min_common_users) & (nbr_sim > min_sim_threshold)
        nbr_idx = nbr_idx[reliable_mask]
        nbr_sim = nbr_sim[reliable_mask]
        nbr_cnt = nbr_cnt[reliable_mask]
        if len(nbr_idx) == 0:
            preds[t] = baseline_fallback[t]
            fallback_flags[t] = 1
            continue
        nbr_sim = nbr_sim * (nbr_cnt / (nbr_cnt + shrinkage))
        rated_map = dict(zip(rated_items, rated_vals))
        sims_used = []
        vals_used = []
        for item_j, sim_val in zip(nbr_idx, nbr_sim):
            if item_j in rated_map:
                sims_used.append(float(sim_val))
                vals_used.append(float(rated_map[item_j]))
        if len(sims_used) < min_neighbors_for_pred:
            preds[t] = baseline_fallback[t]
            fallback_flags[t] = 1
            continue
        sims_used = np.asarray(sims_used, dtype=np.float32)
        vals_used = np.asarray(vals_used, dtype=np.float32)
        denom = np.sum(np.abs(sims_used))
        if denom <= 1e-12:
            preds[t] = baseline_fallback[t]
            fallback_flags[t] = 1
            continue
        residual = np.sum(sims_used * vals_used) / denom
        preds[t] = np.clip(user_means_raw_vec[u] + residual, 1.0, 5.0)
        used_neighbor_counts[t] = len(sims_used)
    return preds.astype(np.float32), used_neighbor_counts, fallback_flags

item_neighbors_common_a_fit = compute_neighbor_co_counts(model_a_fit_artifacts["B"], item_neighbors_idx_a_fit)
print("item_neighbors_common_a_fit shape:", item_neighbors_common_a_fit.shape)
print("Example common counts             :", item_neighbors_common_a_fit[0][:10])

Computing localized co-counts (lazy evaluation)...
  Processed 25000/166990 items...
  Processed 50000/166990 items...
  Processed 75000/166990 items...
  Processed 100000/166990 items...
  Processed 125000/166990 items...
  Processed 150000/166990 items...
Lazy co-counts finished in 116.0s
item_neighbors_common_a_fit shape: (166990, 20)
Example common counts             : [1 1 1 1 1 1 1 1 1 1]


In [11]:
# ============================================
# CELL 10 ? TUNE MODEL A ON val_fit + SAVE OOF
# ============================================

print("=" * 70)
print("MODEL A ? TUNE ON val_fit")
print("=" * 70)

val_a_known_mask = val_fit[COL_USER].isin(user_enc_a_fit) & val_fit[COL_ITEM].isin(item_enc_a_fit)
result_base_a_oof = evaluate_predictions(y_val_true_oof, baseline_val_preds_oof, name="baseline_on_val_fit_model_a")
print_eval(result_base_a_oof)

topk_grid = [20, 50, 100]
min_common_grid = [2, 5, 10]
min_neighbors_grid = [1]
shrinkage_grid = [10.0, 25.0, 50.0]
min_sim_grid = [0.0, 0.02, 0.05]
results_a_grid = []

for topk in topk_grid:
    for min_common in min_common_grid:
        for min_nbr in min_neighbors_grid:
            for shrink in shrinkage_grid:
                for min_sim in min_sim_grid:
                    preds_tmp = baseline_val_preds_oof.copy()
                    used_tmp_full = np.zeros(len(val_fit), dtype=np.int32)
                    fb_tmp_full = np.ones(len(val_fit), dtype=np.int8)
                    if val_a_known_mask.any():
                        val_known_df = val_fit.loc[val_a_known_mask].copy()
                        u_idx_tmp = val_known_df[COL_USER].map(user_enc_a_fit).astype(np.int32).values
                        i_idx_tmp = val_known_df[COL_ITEM].map(item_enc_a_fit).astype(np.int32).values
                        pred_known, used_known, fb_known = predict_item_item_cf_fixed(val_known_df, u_idx_tmp, i_idx_tmp, model_a_fit_artifacts["M_user_centered"], user_means_raw_a_fit, item_neighbors_idx_a_fit, item_neighbors_sim_a_fit, item_neighbors_common_a_fit, user_mean_map_fit, item_mean_map_fit, GLOBAL_MEAN_FIT, topk=topk, min_common_users=min_common, min_neighbors_for_pred=min_nbr, shrinkage=shrink, min_sim_threshold=min_sim)
                        preds_tmp[val_a_known_mask.to_numpy()] = pred_known
                        used_tmp_full[val_a_known_mask.to_numpy()] = used_known
                        fb_tmp_full[val_a_known_mask.to_numpy()] = fb_known
                    res_tmp = evaluate_predictions(y_val_true_oof, preds_tmp, name=f"model_a_topk{topk}_mc{min_common}_mn{min_nbr}_sh{shrink}_ms{min_sim}")
                    results_a_grid.append({"topk": int(topk), "min_common_users": int(min_common), "min_neighbors_for_pred": int(min_nbr), "shrinkage": float(shrink), "min_sim_threshold": float(min_sim), "rmse": float(res_tmp["rmse"]), "mae": float(res_tmp["mae"]), "n": int(res_tmp["n"]), "known_coverage": float(val_a_known_mask.mean()), "avg_used_neighbors": float(np.mean(used_tmp_full)), "median_used_neighbors": float(np.median(used_tmp_full)), "fallback_rate": float(np.mean(fb_tmp_full))})
                    print(f"topk={topk:<3d} | mc={min_common} | mn={min_nbr} | sh={shrink:<4.1f} | ms={min_sim:<4.2f} | OOF RMSE={res_tmp['rmse']:.6f} | OOF MAE={res_tmp['mae']:.6f}")

results_a_grid_df = pd.DataFrame(results_a_grid).sort_values(["rmse", "mae", "fallback_rate", "topk", "min_common_users", "min_neighbors_for_pred", "shrinkage", "min_sim_threshold"], ascending=[True, True, True, True, True, True, True, True]).reset_index(drop=True)
display(results_a_grid_df)

best_row_a = results_a_grid_df.iloc[0]
best_topk = int(best_row_a["topk"])
best_mc = int(best_row_a["min_common_users"])
best_mn = int(best_row_a["min_neighbors_for_pred"])
best_sh = float(best_row_a["shrinkage"])
best_ms = float(best_row_a["min_sim_threshold"])

model_a_val_preds_oof = baseline_val_preds_oof.copy()
model_a_val_used_neighbors_oof = np.zeros(len(val_fit), dtype=np.int32)
model_a_val_fallback_flags_oof = np.ones(len(val_fit), dtype=np.int8)
if val_a_known_mask.any():
    val_known_df = val_fit.loc[val_a_known_mask].copy()
    u_idx_val_a = val_known_df[COL_USER].map(user_enc_a_fit).astype(np.int32).values
    i_idx_val_a = val_known_df[COL_ITEM].map(item_enc_a_fit).astype(np.int32).values
    pred_known, used_known, fb_known = predict_item_item_cf_fixed(val_known_df, u_idx_val_a, i_idx_val_a, model_a_fit_artifacts["M_user_centered"], user_means_raw_a_fit, item_neighbors_idx_a_fit, item_neighbors_sim_a_fit, item_neighbors_common_a_fit, user_mean_map_fit, item_mean_map_fit, GLOBAL_MEAN_FIT, topk=best_topk, min_common_users=best_mc, min_neighbors_for_pred=best_mn, shrinkage=best_sh, min_sim_threshold=best_ms)
    model_a_val_preds_oof[val_a_known_mask.to_numpy()] = pred_known
    model_a_val_used_neighbors_oof[val_a_known_mask.to_numpy()] = used_known
    model_a_val_fallback_flags_oof[val_a_known_mask.to_numpy()] = fb_known

model_a_val_result_oof = evaluate_predictions(y_val_true_oof, model_a_val_preds_oof, name="model_a_val_oof")
print_eval(model_a_val_result_oof)
np.save(STAGE3_DIR / "model_a_val_preds_oof.npy", model_a_val_preds_oof)
np.save(STAGE3_DIR / "model_a_val_used_neighbors_oof.npy", model_a_val_used_neighbors_oof)
np.save(STAGE3_DIR / "model_a_val_fallback_flags_oof.npy", model_a_val_fallback_flags_oof)
results_a_grid_df.to_csv(STAGE3_DIR / "model_a_tuning.csv", index=False)

MODEL A ? TUNE ON val_fit
Model: baseline_on_val_fit_model_a
N    : 449825
RMSE : 1.094414
MAE  : 0.792196
------------------------------------------------------------
star_1_rmse   : 3.101513385772705
star_1_mae    : 3.0546438694000244
star_1_n      : 24386
star_2_rmse   : 2.181936264038086
star_2_mae    : 2.1269261837005615
star_2_n      : 18751
star_3_rmse   : 1.2853533029556274
star_3_mae    : 1.2179089784622192
star_3_n      : 32659
star_4_rmse   : 0.4962732791900635
star_4_mae    : 0.4226570427417755
star_4_n      : 62459
star_5_rmse   : 0.6835676431655884
star_5_mae    : 0.5642480254173279
star_5_n      : 311570
topk=20  | mc=2 | mn=1 | sh=10.0 | ms=0.00 | OOF RMSE=1.094548 | OOF MAE=0.792007
topk=20  | mc=2 | mn=1 | sh=10.0 | ms=0.02 | OOF RMSE=1.094535 | OOF MAE=0.792017
topk=20  | mc=2 | mn=1 | sh=10.0 | ms=0.05 | OOF RMSE=1.094525 | OOF MAE=0.792064
topk=20  | mc=2 | mn=1 | sh=25.0 | ms=0.00 | OOF RMSE=1.094548 | OOF MAE=0.792007
topk=20  | mc=2 | mn=1 | sh=25.0 | ms=0.02 | 

,topk,min_common_users,min_neighbors_for_pred,shrinkage,min_sim_threshold,rmse,mae,n,known_coverage,avg_used_neighbors,median_used_neighbors,fallback_rate
0,20,10,1,10.0,0.0,1.094391,0.792080,449825,0.831032,0.000427,0.0,0.999584
1,20,10,1,25.0,0.0,1.094391,0.792080,449825,0.831032,0.000427,0.0,0.999584
2,20,10,1,50.0,0.0,1.094391,0.792080,449825,0.831032,0.000427,0.0,0.999584
3,50,10,1,10.0,0.0,1.094391,0.792080,449825,0.831032,0.000427,0.0,0.999584
4,50,10,1,25.0,0.0,1.094391,0.792080,449825,0.831032,0.000427,0.0,0.999584
...,...,...,...,...,...,...,...,...,...,...,...,...
76,20,2,1,50.0,0.0,1.094548,0.792007,449825,0.831032,0.001463,0.0,0.998619
77,50,2,1,25.0,0.0,1.094548,0.792007,449825,0.831032,0.001463,0.0,0.998619
78,50,2,1,50.0,0.0,1.094548,0.792007,449825,0.831032,0.001463,0.0,0.998619
79,100,2,1,25.0,0.0,1.094548,0.792007,449825,0.831032,0.001463,0.0,0.998619


Model: model_a_val_oof
N    : 449825
RMSE : 1.094391
MAE  : 0.79208
------------------------------------------------------------
star_1_rmse   : 3.1013641357421875
star_1_mae    : 3.054328680038452
star_1_n      : 24386
star_2_rmse   : 2.181816339492798
star_2_mae    : 2.126626491546631
star_2_n      : 18751
star_3_rmse   : 1.2853423357009888
star_3_mae    : 1.2177900075912476
star_3_n      : 32659
star_4_rmse   : 0.4964609444141388
star_4_mae    : 0.4227665662765503
star_4_n      : 62459
star_5_rmse   : 0.6835636496543884
star_5_mae    : 0.5641139149665833
star_5_n      : 311570


In [12]:
# ============================================
# CELL 11 ? REFIT MODEL A ON FULL TRAIN + FINAL SAVE
# ============================================

print("=" * 70)
print("MODEL A ? REFIT BEST CONFIG ON FULL TRAIN")
print("=" * 70)
print("Best Model A config")
print("-" * 60)
print("topk                  :", best_topk)
print("min_common_users      :", best_mc)
print("min_neighbors_for_pred:", best_mn)
print("shrinkage             :", best_sh)
print("min_sim_threshold     :", best_ms)

model_a_full_artifacts = build_item_item_cf_artifacts(train)
user_enc_a = model_a_full_artifacts["user_enc"]
item_enc_a = model_a_full_artifacts["item_enc"]
user_means_raw_a = model_a_full_artifacts["user_means_raw"]
M_user_centered_a = model_a_full_artifacts["M_user_centered"]
item_neighbors_idx_a, item_neighbors_sim_a = precompute_item_neighbors(model_a_full_artifacts["M_user_centered"], max_k=MAX_K)
item_neighbors_common_a = compute_neighbor_co_counts(model_a_full_artifacts["B"], item_neighbors_idx_a)

known_mask_test_a = test_warm[COL_USER].isin(user_enc_a) & test_warm[COL_ITEM].isin(item_enc_a)
model_a_preds = y_pred_base.copy()
model_a_used_neighbors = np.zeros(len(test_warm), dtype=np.int32)
model_a_fallback_flags = np.ones(len(test_warm), dtype=np.int8)
if known_mask_test_a.any():
    test_known_df_a = test_warm.loc[known_mask_test_a].copy()
    u_idx_test_a = test_known_df_a[COL_USER].map(user_enc_a).astype(np.int32).values
    i_idx_test_a = test_known_df_a[COL_ITEM].map(item_enc_a).astype(np.int32).values
    pred_known, used_known, fb_known = predict_item_item_cf_fixed(test_known_df_a, u_idx_test_a, i_idx_test_a, M_user_centered_a, user_means_raw_a, item_neighbors_idx_a, item_neighbors_sim_a, item_neighbors_common_a, user_mean_map, item_mean_map, GLOBAL_MEAN, topk=best_topk, min_common_users=best_mc, min_neighbors_for_pred=best_mn, shrinkage=best_sh, min_sim_threshold=best_ms)
    model_a_preds[known_mask_test_a.to_numpy()] = pred_known
    model_a_used_neighbors[known_mask_test_a.to_numpy()] = used_known
    model_a_fallback_flags[known_mask_test_a.to_numpy()] = fb_known

assert len(model_a_preds) == len(test_warm), "Model A final predictions must align with test_warm."
result_a_best = evaluate_predictions(y_true_warm, model_a_preds, name="model_a_item_item_cf_full_train")
result_base_a = evaluate_predictions(y_true_warm, y_pred_base, name="baseline_on_test_warm")
print_eval(result_base_a)
print_eval(result_a_best)

rmse_gain_a = result_base_a["rmse"] - result_a_best["rmse"]
mae_gain_a = result_base_a["mae"] - result_a_best["mae"]
model_a_summary = {"best_topk": int(best_topk), "best_min_common_users": int(best_mc), "best_min_neighbors_for_pred": int(best_mn), "best_shrinkage": float(best_sh), "best_min_sim_threshold": float(best_ms), "val_fit_rmse": float(model_a_val_result_oof["rmse"]), "val_fit_mae": float(model_a_val_result_oof["mae"]), "val_fit_known_coverage": float(val_a_known_mask.mean()), "test_warm_rmse": float(result_a_best["rmse"]), "test_warm_mae": float(result_a_best["mae"]), "rmse_gain_vs_baseline_test_warm": float(rmse_gain_a), "mae_gain_vs_baseline_test_warm": float(mae_gain_a), "avg_used_neighbors": float(np.mean(model_a_used_neighbors)), "median_used_neighbors": float(np.median(model_a_used_neighbors)), "fallback_rate": float(np.mean(model_a_fallback_flags)), "test_known_coverage": float(known_mask_test_a.mean()), "test_warm_used_only_for_final_evaluation": True, "per_star": result_a_best["per_star"]}

np.save(STAGE3_DIR / "model_a_preds.npy", model_a_preds)
np.save(STAGE3_DIR / "model_a_used_neighbors.npy", model_a_used_neighbors)
np.save(STAGE3_DIR / "model_a_fallback_flags.npy", model_a_fallback_flags)
np.save(STAGE3_DIR / "model_a_neighbor_idx.npy", item_neighbors_idx_a)
np.save(STAGE3_DIR / "model_a_neighbor_sim.npy", item_neighbors_sim_a)
np.save(STAGE3_DIR / "model_a_neighbor_common.npy", item_neighbors_common_a)
with open(STAGE3_DIR / "model_a_result.json", "w") as f:
    json.dump(to_jsonable(result_a_best), f, indent=2)
with open(STAGE3_DIR / "model_a_summary.json", "w") as f:
    json.dump(to_jsonable(model_a_summary), f, indent=2)

MODEL A ? REFIT BEST CONFIG ON FULL TRAIN
Best Model A config
------------------------------------------------------------
topk                  : 20
min_common_users      : 10
min_neighbors_for_pred: 1
shrinkage             : 10.0
min_sim_threshold     : 0.0
Computing localized co-counts (lazy evaluation)...
  Processed 25000/169241 items...
  Processed 50000/169241 items...
  Processed 75000/169241 items...
  Processed 100000/169241 items...
  Processed 125000/169241 items...
  Processed 150000/169241 items...
Lazy co-counts finished in 123.4s
Model: baseline_on_test_warm
N    : 935578
RMSE : 1.144186
MAE  : 0.855168
------------------------------------------------------------
star_1_rmse   : 3.215768337249756
star_1_mae    : 3.2064449787139893
star_1_n      : 60306
star_2_rmse   : 2.2479584217071533
star_2_mae    : 2.236931562423706
star_2_n      : 44048
star_3_rmse   : 1.2934361696243286
star_3_mae    : 1.2766953706741333
star_3_n      : 70544
star_4_rmse   : 0.3738032281398773
sta

In [13]:
# ============================================
# CELL 12 ? MODEL B PREP (MF WITH SGD)
# ============================================
import numpy as np
import pandas as pd

print("=" * 70)
print("MODEL B ? MATRIX FACTORIZATION WITH SGD")
print("=" * 70)
assert "user_enc_a_fit" in globals(), "Run Model A tuning cells first."
assert "user_enc_a" in globals(), "Run Model A full-train refit cell first."

train_b_fit_encoded = train_fit.copy()
train_b_fit_encoded["user_idx"] = train_b_fit_encoded[COL_USER].map(user_enc_a_fit)
train_b_fit_encoded["item_idx"] = train_b_fit_encoded[COL_ITEM].map(item_enc_a_fit)
train_b_fit_encoded = train_b_fit_encoded.dropna(subset=["user_idx", "item_idx"]).copy()
train_b_fit_encoded["user_idx"] = train_b_fit_encoded["user_idx"].astype(np.int32)
train_b_fit_encoded["item_idx"] = train_b_fit_encoded["item_idx"].astype(np.int32)

u_train_fit_b = train_b_fit_encoded["user_idx"].values.astype(np.int32)
i_train_fit_b = train_b_fit_encoded["item_idx"].values.astype(np.int32)
r_train_fit_b = train_b_fit_encoded[COL_RATING].astype(np.float32).values
item_mean_fit_arr_b = train_b_fit_encoded[COL_ITEM].map(item_mean_map_fit).fillna(GLOBAL_MEAN_FIT).astype(np.float32).values
baseline_train_fit_b = 0.5 * (user_means_raw_a_fit[u_train_fit_b] + item_mean_fit_arr_b)
y_train_fit_b = (r_train_fit_b - baseline_train_fit_b).astype(np.float32)

val_b_known_mask = val_fit[COL_USER].isin(user_enc_a_fit) & val_fit[COL_ITEM].isin(item_enc_a_fit)

train_b_full_encoded = train.copy()
train_b_full_encoded["user_idx"] = train_b_full_encoded[COL_USER].map(user_enc_a)
train_b_full_encoded["item_idx"] = train_b_full_encoded[COL_ITEM].map(item_enc_a)
train_b_full_encoded = train_b_full_encoded.dropna(subset=["user_idx", "item_idx"]).copy()
train_b_full_encoded["user_idx"] = train_b_full_encoded["user_idx"].astype(np.int32)
train_b_full_encoded["item_idx"] = train_b_full_encoded["item_idx"].astype(np.int32)

u_train_b = train_b_full_encoded["user_idx"].values.astype(np.int32)
i_train_b = train_b_full_encoded["item_idx"].values.astype(np.int32)
r_train_b = train_b_full_encoded[COL_RATING].astype(np.float32).values
item_mean_train_arr_b_full = train_b_full_encoded[COL_ITEM].map(item_mean_map).fillna(GLOBAL_MEAN).astype(np.float32).values
baseline_train_b = 0.5 * (user_means_raw_a[u_train_b] + item_mean_train_arr_b_full)
y_train_b = (r_train_b - baseline_train_b).astype(np.float32)

known_mask_test_b = test_warm[COL_USER].isin(user_enc_a) & test_warm[COL_ITEM].isin(item_enc_a)
y_true_b = y_true_warm.copy()

print("Train-fit triplets:", len(train_b_fit_encoded))
print("Full-train triplets:", len(train_b_full_encoded))
print("Validation known coverage for Model B:", round(float(val_b_known_mask.mean()), 6))
print("Test known coverage for Model B:", round(float(known_mask_test_b.mean()), 6))
print("Residual target stats on train_fit:")
print(pd.Series(y_train_fit_b).describe())


MODEL B ? MATRIX FACTORIZATION WITH SGD
Train-fit triplets: 4048424
Full-train triplets: 4498249
Validation known coverage for Model B: 0.831032
Test known coverage for Model B: 1.0
Residual target stats on train_fit:
count    4.048424e+06
mean    -1.320274e-02
std      9.488032e-01
min     -4.974719e+00
25%     -3.009856e-01
50%      2.958333e-01
75%      5.586957e-01
max      2.857143e+00
dtype: float64


In [14]:
# ============================================
# CELL 13 — MODEL B CORE FUNCTIONS
# ============================================

def predict_mf_residual(u_idx, i_idx, P, Q):
    """
    residual prediction on user-centered target:
      residual_hat = dot(P[u], Q[i])
    """
    return np.sum(P[u_idx] * Q[i_idx], axis=1).astype(np.float32)


# ============================================
# CELL 13 — MODEL B CORE FUNCTIONS (FIXED)
# ============================================

def predict_model_b(
    df_subset,
    u_idx,
    i_idx,
    P, Q,
    bu, bi,
    user_means_raw_vec,
    item_mean_map,
    global_mean
):
    item_means = (
        df_subset[COL_ITEM].map(item_mean_map)
        .fillna(global_mean).astype(np.float32).values
    )
    baseline = 0.5 * (user_means_raw_vec[u_idx] + item_means)
    residual_hat = (
        bu[u_idx] + bi[i_idx]
        + np.sum(P[u_idx] * Q[i_idx], axis=1)
    ).astype(np.float32)
    preds = np.clip(baseline + residual_hat, 1.0, 5.0).astype(np.float32)
    return preds, residual_hat


from numba import njit, prange
 
@njit(parallel=False, cache=True)
def _sgd_epoch_numba(order, u_idx, i_idx, y_residual,
                     P, Q, bu, bi, lr, reg):
    """
    Single SGD epoch — compiled by Numba.
    parallel=False vì update P/Q có data dependency per user/item.
    cache=True để lần chạy sau không cần compile lại.
    
    Returns: sum of squared errors (để tính epoch RMSE)
    """
    se_sum = 0.0
    n_factors = P.shape[1]
    
    for idx in range(len(order)):
        t = order[idx]
        u = u_idx[t]
        i = i_idx[t]
        y = y_residual[t]
        
        # Compute prediction
        dot = 0.0
        for f in range(n_factors):
            dot += P[u, f] * Q[i, f]
        pred = bu[u] + bi[i] + dot
        
        err = y - pred
        se_sum += err * err
        
        # Update biases
        bu[u] += lr * (err - reg * bu[u])
        bi[i] += lr * (err - reg * bi[i])
        
        # Update factors
        for f in range(n_factors):
            pu_f = P[u, f]
            qi_f = Q[i, f]
            P[u, f] += lr * (err * qi_f - reg * pu_f)
            Q[i, f] += lr * (err * pu_f - reg * qi_f)
    
    return se_sum
 
 
def train_mf_sgd(                     # Tên giống hệt để không cần sửa Cell 15/16
    u_idx, i_idx, y_residual,
    n_users, n_items,
    n_factors=50,
    lr=0.01, reg=0.05,
    n_epochs=20,
    shuffle=True, random_state=42,
    verbose=True
):
    """
    SVD-style MF với Numba backend.
    Interface giống hệt version cũ — drop-in replacement.
    
    Lần đầu chạy sẽ mất ~30s để Numba compile.
    Các lần sau (cache=True) compile ngay lập tức.
    """
    rng = np.random.default_rng(random_state)
    
    P  = rng.normal(0, 0.01, (n_users, n_factors)).astype(np.float32)
    Q  = rng.normal(0, 0.01, (n_items, n_factors)).astype(np.float32)
    bu = np.zeros(n_users, dtype=np.float32)
    bi = np.zeros(n_items, dtype=np.float32)
    
    n = len(y_residual)
    order = np.arange(n, dtype=np.int32)
    
    # Warm-up compile: chạy 1 sample để trigger JIT
    if verbose:
        print("Numba JIT compiling (first run ~30s, cached after)...")
    _sgd_epoch_numba(
        order[:1],
        u_idx[:1].astype(np.int32),
        i_idx[:1].astype(np.int32),
        y_residual[:1].astype(np.float32),
        P, Q, bu, bi,
        np.float32(lr), np.float32(reg)
    )
    if verbose:
        print("Compiled. Starting training...")
    
    history = []
    cur_lr = np.float32(lr)
    
    for epoch in range(1, n_epochs + 1):
        if shuffle:
            rng.shuffle(order)
        
        se_sum = _sgd_epoch_numba(
            order,
            u_idx.astype(np.int32),
            i_idx.astype(np.int32),
            y_residual.astype(np.float32),
            P, Q, bu, bi,
            cur_lr, np.float32(reg)
        )
        
        epoch_rmse = float(np.sqrt(se_sum / n))
        cur_lr = np.float32(cur_lr * 0.95)
        history.append({"epoch": epoch, "train_residual_rmse": epoch_rmse})
        
        if verbose:
            print(f"Epoch {epoch:02d}/{n_epochs} | residual RMSE={epoch_rmse:.6f}")
    
    return P, Q, bu, bi, history

In [15]:
# ============================================
# CELL 14 ? MODEL B TRAIN_FIT / val_fit ARRAYS
# ============================================

u_fit_b = u_train_fit_b.astype(np.int32)
i_fit_b = i_train_fit_b.astype(np.int32)
y_fit_b = y_train_fit_b.astype(np.float32)

res_val_base = evaluate_predictions(y_val_true_oof, baseline_val_preds_oof, name="baseline_on_val_fit_model_b")
print_eval(res_val_base)
print("Train-fit size :", len(u_fit_b))
print("Val-fit size   :", len(val_fit))
print("Val date range :", date_range_str(val_fit))


Model: baseline_on_val_fit_model_b
N    : 449825
RMSE : 1.094414
MAE  : 0.792196
------------------------------------------------------------
star_1_rmse   : 3.101513385772705
star_1_mae    : 3.0546438694000244
star_1_n      : 24386
star_2_rmse   : 2.181936264038086
star_2_mae    : 2.1269261837005615
star_2_n      : 18751
star_3_rmse   : 1.2853533029556274
star_3_mae    : 1.2179089784622192
star_3_n      : 32659
star_4_rmse   : 0.4962732791900635
star_4_mae    : 0.4226570427417755
star_4_n      : 62459
star_5_rmse   : 0.6835676431655884
star_5_mae    : 0.5642480254173279
star_5_n      : 311570
Train-fit size : 4048424
Val-fit size   : 449825
Val date range : 2017-01-12 00:00:00 -> 2017-05-15 00:00:00


In [16]:
# ============================================
# CELL 15 ? TUNE MODEL B ON val_fit + SAVE OOF
# ============================================

factor_grid = [20, 50]
lr_grid     = [0.01, 0.02]
reg_grid    = [0.02, 0.05]
epoch_grid  = [5, 10, 20, 30]

results_b_grid = []
best_model_b = None
best_model_b_key = (float("inf"), float("inf"))

for n_factors in factor_grid:
    for lr in lr_grid:
        for reg in reg_grid:
            for n_epochs in epoch_grid:
                P_tmp, Q_tmp, bu_tmp, bi_tmp, hist_tmp = train_mf_sgd(u_idx=u_fit_b, i_idx=i_fit_b, y_residual=y_fit_b, n_users=len(user_enc_a_fit), n_items=len(item_enc_a_fit), n_factors=n_factors, lr=lr, reg=reg, n_epochs=n_epochs, verbose=False)
                preds_tmp = baseline_val_preds_oof.copy()
                residual_tmp_full = np.zeros(len(val_fit), dtype=np.float32)
                if val_b_known_mask.any():
                    val_known_df_b = val_fit.loc[val_b_known_mask].copy()
                    u_val_b = val_known_df_b[COL_USER].map(user_enc_a_fit).astype(np.int32).values
                    i_val_b = val_known_df_b[COL_ITEM].map(item_enc_a_fit).astype(np.int32).values
                    pred_known, residual_known = predict_model_b(val_known_df_b, u_val_b, i_val_b, P_tmp, Q_tmp, bu_tmp, bi_tmp, user_means_raw_a_fit, item_mean_map_fit, GLOBAL_MEAN_FIT)
                    preds_tmp[val_b_known_mask.to_numpy()] = pred_known
                    residual_tmp_full[val_b_known_mask.to_numpy()] = residual_known
                res_val_tmp = evaluate_predictions(y_val_true_oof, preds_tmp, name=f"model_b_k{n_factors}_lr{lr}_reg{reg}_ep{n_epochs}")
                train_resid_rmse_last = hist_tmp[-1]["train_residual_rmse"]
                results_b_grid.append({"n_factors": int(n_factors), "lr": float(lr), "reg": float(reg), "n_epochs": int(n_epochs), "val_rmse": float(res_val_tmp["rmse"]), "val_mae": float(res_val_tmp["mae"]), "train_residual_rmse_last": float(train_resid_rmse_last), "known_coverage": float(val_b_known_mask.mean())})
                print(f"VAL RMSE={res_val_tmp['rmse']:.6f} | VAL MAE={res_val_tmp['mae']:.6f} | train_resid_RMSE={train_resid_rmse_last:.6f}")
                cur_key = (float(res_val_tmp["rmse"]), float(res_val_tmp["mae"]))
                if cur_key < best_model_b_key:
                    best_model_b_key = cur_key
                    best_model_b = {"P": P_tmp.copy(), "Q": Q_tmp.copy(), "bu": bu_tmp.copy(), "bi": bi_tmp.copy(), "history": hist_tmp, "config": {"n_factors": int(n_factors), "lr": float(lr), "reg": float(reg), "n_epochs": int(n_epochs)}, "val_result": res_val_tmp, "val_preds": preds_tmp.copy(), "val_residual": residual_tmp_full.copy()}

results_b_grid_df = pd.DataFrame(results_b_grid).sort_values(["val_rmse", "val_mae", "n_factors", "reg", "lr", "n_epochs"]).reset_index(drop=True)
display(results_b_grid_df)

best_cfg_b = best_model_b["config"]
model_b_val_preds_oof = best_model_b["val_preds"].astype(np.float32)
model_b_val_residual_oof = best_model_b["val_residual"].astype(np.float32)
model_b_val_result_oof = evaluate_predictions(y_val_true_oof, model_b_val_preds_oof, name="model_b_val_oof")
print("\nBest config on val_fit:")
print(best_cfg_b)
print("Best val RMSE:", model_b_val_result_oof["rmse"])
print("Baseline val RMSE:", res_val_base["rmse"])
print_eval(model_b_val_result_oof)

np.save(STAGE3_DIR / "model_b_val_preds_oof.npy", model_b_val_preds_oof)
np.save(STAGE3_DIR / "model_b_val_residual_oof.npy", model_b_val_residual_oof)
results_b_grid_df.to_csv(STAGE3_DIR / "model_b_tuning.csv", index=False)


VAL RMSE=1.094427 | VAL MAE=0.772119 | train_resid_RMSE=0.927860
VAL RMSE=1.097652 | VAL MAE=0.770305 | train_resid_RMSE=0.915229
VAL RMSE=1.102347 | VAL MAE=0.769636 | train_resid_RMSE=0.902633
VAL RMSE=1.105151 | VAL MAE=0.769744 | train_resid_RMSE=0.893347
VAL RMSE=1.094453 | VAL MAE=0.772369 | train_resid_RMSE=0.927994
VAL RMSE=1.097645 | VAL MAE=0.770567 | train_resid_RMSE=0.915588
VAL RMSE=1.102273 | VAL MAE=0.769883 | train_resid_RMSE=0.904529
VAL RMSE=1.105040 | VAL MAE=0.769970 | train_resid_RMSE=0.899263
VAL RMSE=1.099187 | VAL MAE=0.770193 | train_resid_RMSE=0.921945
VAL RMSE=1.105910 | VAL MAE=0.770195 | train_resid_RMSE=0.904355
VAL RMSE=1.114076 | VAL MAE=0.771590 | train_resid_RMSE=0.844318
VAL RMSE=1.118934 | VAL MAE=0.773188 | train_resid_RMSE=0.785050
VAL RMSE=1.099166 | VAL MAE=0.770454 | train_resid_RMSE=0.922256
VAL RMSE=1.105789 | VAL MAE=0.770423 | train_resid_RMSE=0.907968
VAL RMSE=1.113732 | VAL MAE=0.771655 | train_resid_RMSE=0.886771
VAL RMSE=1.117869 | VAL M

,n_factors,lr,reg,n_epochs,val_rmse,val_mae,train_residual_rmse_last,known_coverage
0,50,0.01,0.02,5,1.094309,0.772204,0.927653,0.831032
1,50,0.01,0.05,5,1.094336,0.772454,0.927824,0.831032
2,20,0.01,0.02,5,1.094427,0.772119,0.927860,0.831032
3,20,0.01,0.05,5,1.094453,0.772369,0.927994,0.831032
4,50,0.01,0.05,10,1.097572,0.770500,0.915188,0.831032
5,50,0.01,0.02,10,1.097576,0.770237,0.914619,0.831032
6,20,0.01,0.05,10,1.097645,0.770567,0.915588,0.831032
7,20,0.01,0.02,10,1.097652,0.770305,0.915229,0.831032
8,50,0.02,0.05,5,1.098972,0.770575,0.921872,0.831032
9,50,0.02,0.02,5,1.098991,0.770317,0.921358,0.831032



Best config on val_fit:
{'n_factors': 50, 'lr': 0.01, 'reg': 0.02, 'n_epochs': 5}
Best val RMSE: 1.0943094491958618
Baseline val RMSE: 1.094414234161377
Model: model_b_val_oof
N    : 449825
RMSE : 1.094309
MAE  : 0.772204
------------------------------------------------------------
star_1_rmse   : 3.049503803253174
star_1_mae    : 2.982616424560547
star_1_n      : 24386
star_2_rmse   : 2.150444746017456
star_2_mae    : 2.073028564453125
star_2_n      : 18751
star_3_rmse   : 1.2832754850387573
star_3_mae    : 1.1947704553604126
star_3_n      : 32659
star_4_rmse   : 0.5494146943092346
star_4_mae    : 0.46256697177886963
star_4_n      : 62459
star_5_rmse   : 0.6997161507606506
star_5_mae    : 0.538690447807312
star_5_n      : 311570


In [17]:
# ============================================
# CELL 16 ? REFIT BEST MODEL B ON FULL TRAIN
# ============================================

print("Refitting best Model B on full train...")
print(best_cfg_b)

P_b, Q_b, bu_b, bi_b, hist_b = train_mf_sgd(u_idx=u_train_b, i_idx=i_train_b, y_residual=y_train_b, n_users=len(user_enc_a), n_items=len(item_enc_a), n_factors=best_cfg_b["n_factors"], lr=best_cfg_b["lr"], reg=best_cfg_b["reg"], n_epochs=best_cfg_b["n_epochs"], shuffle=True, random_state=42, verbose=True)

model_b_preds = y_pred_base.copy()
model_b_residual = np.zeros(len(test_warm), dtype=np.float32)
if known_mask_test_b.any():
    test_known_df_b = test_warm.loc[known_mask_test_b].copy()
    u_idx_test_b = test_known_df_b[COL_USER].map(user_enc_a).astype(np.int32).values
    i_idx_test_b = test_known_df_b[COL_ITEM].map(item_enc_a).astype(np.int32).values
    pred_known, residual_known = predict_model_b(test_known_df_b, u_idx_test_b, i_idx_test_b, P_b, Q_b, bu_b, bi_b, user_means_raw_a, item_mean_map, GLOBAL_MEAN)
    model_b_preds[known_mask_test_b.to_numpy()] = pred_known
    model_b_residual[known_mask_test_b.to_numpy()] = residual_known

assert len(model_b_preds) == len(test_warm), "Model B final predictions must align with test_warm."
result_b = evaluate_predictions(y_true_warm, model_b_preds, name=(f"model_b_mf_sgd_k{best_cfg_b['n_factors']}_lr{best_cfg_b['lr']}_reg{best_cfg_b['reg']}_ep{best_cfg_b['n_epochs']}"))
result_base_b = evaluate_predictions(y_true_warm, y_pred_base, name="baseline_on_test_warm")
print_eval(result_base_b)
print_eval(result_b)
rmse_gain_b = result_base_b["rmse"] - result_b["rmse"]
mae_gain_b = result_base_b["mae"] - result_b["mae"]
print("RMSE gain over baseline test_warm:", rmse_gain_b)
print("MAE  gain over baseline test_warm:", mae_gain_b)
print("\nResidual stats on full test_warm:")
print(pd.Series(model_b_residual).describe())


Refitting best Model B on full train...
{'n_factors': 50, 'lr': 0.01, 'reg': 0.02, 'n_epochs': 5}
Numba JIT compiling (first run ~30s, cached after)...
Compiled. Starting training...
Epoch 01/5 | residual RMSE=0.968395
Epoch 02/5 | residual RMSE=0.956948
Epoch 03/5 | residual RMSE=0.949351
Epoch 04/5 | residual RMSE=0.943510
Epoch 05/5 | residual RMSE=0.938828
Model: baseline_on_test_warm
N    : 935578
RMSE : 1.144186
MAE  : 0.855168
------------------------------------------------------------
star_1_rmse   : 3.215768337249756
star_1_mae    : 3.2064449787139893
star_1_n      : 60306
star_2_rmse   : 2.2479584217071533
star_2_mae    : 2.236931562423706
star_2_n      : 44048
star_3_rmse   : 1.2934361696243286
star_3_mae    : 1.2766953706741333
star_3_n      : 70544
star_4_rmse   : 0.3738032281398773
star_4_mae    : 0.3362816274166107
star_4_n      : 126808
star_5_rmse   : 0.6189844608306885
star_5_mae    : 0.5923428535461426
star_5_n      : 633872
Model: model_b_mf_sgd_k50_lr0.01_reg0.02_

In [18]:
# ============================================
# CELL 17 ? SAVE MODEL B
# ============================================

model_b_summary = {"best_config": to_jsonable(best_cfg_b), "val_fit_rmse": float(model_b_val_result_oof["rmse"]), "val_fit_mae": float(model_b_val_result_oof["mae"]), "val_fit_known_coverage": float(val_b_known_mask.mean()), "test_warm_rmse": float(result_b["rmse"]), "test_warm_mae": float(result_b["mae"]), "rmse_gain_vs_baseline_test_warm": float(rmse_gain_b), "mae_gain_vs_baseline_test_warm": float(mae_gain_b), "per_star": result_b["per_star"], "bu_abs_mean": float(np.abs(bu_b).mean()), "bi_abs_mean": float(np.abs(bi_b).mean()), "train_history": hist_b, "test_warm_used_only_for_final_evaluation": True}

np.save(STAGE3_DIR / "model_b_preds.npy", model_b_preds)
np.save(STAGE3_DIR / "model_b_residual.npy", model_b_residual)
np.save(STAGE3_DIR / "model_b_P.npy", P_b)
np.save(STAGE3_DIR / "model_b_Q.npy", Q_b)
np.save(STAGE3_DIR / "model_b_bu.npy", bu_b)
np.save(STAGE3_DIR / "model_b_bi.npy", bi_b)
with open(STAGE3_DIR / "model_b_result.json", "w") as f:
    json.dump(to_jsonable(result_b), f, indent=2)
with open(STAGE3_DIR / "model_b_summary.json", "w") as f:
    json.dump(to_jsonable(model_b_summary), f, indent=2)


In [19]:
# ============================================
# CELL 18 ? MODEL D PREP (ALS IMPLICIT)
# ============================================
"""
Model D uses implicit-feedback ALS.

Leakage rule here:
- Tune only on train_fit -> val_fit.
- Refit on full train only after best config is selected.
- test_warm is final evaluation only.
"""
import json
import warnings
warnings.filterwarnings("ignore")

print("=" * 70)
print("MODEL D ? ALS IMPLICIT FEEDBACK")
print("=" * 70)
print("Stage 2 C_implicit is loaded for reference only:")
print("  C_implicit shape:", C_implicit.shape)
print("  C_implicit nnz  :", C_implicit.nnz)
print("  train_fit rows  :", len(train_fit))
print("  val_fit rows    :", len(val_fit))

result_base_d = evaluate_predictions(y_true_warm, y_pred_base, name="baseline_on_test_warm_model_d")
print_eval(result_base_d)


MODEL D ? ALS IMPLICIT FEEDBACK
Stage 2 C_implicit is loaded for reference only:
  C_implicit shape: (620917, 169241)
  C_implicit nnz  : 4476056
  train_fit rows  : 4048424
  val_fit rows    : 449825
Model: baseline_on_test_warm_model_d
N    : 935578
RMSE : 1.144186
MAE  : 0.855168
------------------------------------------------------------
star_1_rmse   : 3.215768337249756
star_1_mae    : 3.2064449787139893
star_1_n      : 60306
star_2_rmse   : 2.2479584217071533
star_2_mae    : 2.236931562423706
star_2_n      : 44048
star_3_rmse   : 1.2934361696243286
star_3_mae    : 1.2766953706741333
star_3_n      : 70544
star_4_rmse   : 0.3738032281398773
star_4_mae    : 0.3362816274166107
star_4_n      : 126808
star_5_rmse   : 0.6189844608306885
star_5_mae    : 0.5923428535461426
star_5_n      : 633872


In [20]:
# ============================================
# CELL 19 ? ALS CORE IMPLEMENTATION + MATRIX BUILDERS
# ============================================
import implicit

def als_implicit_train(C_csr, n_factors=20, n_iter=10, reg=0.01, alpha=1.0, random_state=42, verbose=True):
    C_scaled = C_csr.copy().astype(np.float64)
    if alpha != 1.0:
        C_scaled.data *= alpha
    model = implicit.als.AlternatingLeastSquares(factors=n_factors, iterations=n_iter, regularization=reg, random_state=random_state, use_gpu=False, calculate_training_loss=verbose)
    model.fit(C_scaled.tocsr(), show_progress=verbose)
    U = np.asarray(model.user_factors, dtype=np.float32)
    V = np.asarray(model.item_factors, dtype=np.float32)
    history = [{"iter": i + 1, "approx_loss": float("nan")} for i in range(n_iter)]
    return U, V, history

def build_implicit_artifacts(train_source_df):
    user_list = train_source_df[COL_USER].drop_duplicates().tolist()
    item_list = train_source_df[COL_ITEM].drop_duplicates().tolist()
    user_enc_local = {user_id: idx for idx, user_id in enumerate(user_list)}
    item_enc_local = {item_id: idx for idx, item_id in enumerate(item_list)}
    rows = train_source_df[COL_USER].map(user_enc_local).astype(np.int32).values
    cols = train_source_df[COL_ITEM].map(item_enc_local).astype(np.int32).values
    vals = np.ones(len(train_source_df), dtype=np.float32)
    C_local = sp.csr_matrix((vals, (rows, cols)), shape=(len(user_list), len(item_list)), dtype=np.float32)
    C_local.sum_duplicates()
    alpha_local = 40.0 if bool((C_local.data <= 1.0).all()) else 1.0
    return {"user_enc": user_enc_local, "item_enc": item_enc_local, "C": C_local, "alpha": float(alpha_local)}

print("ALS helpers ready.")


ALS helpers ready.


In [21]:
# ============================================
# CELL 20 ? MODEL D TRAIN_FIT / VAL_FIT ARTIFACTS
# ============================================

print("=" * 70)
print("MODEL D ? BUILD train_fit ARTIFACTS")
print("=" * 70)

model_d_fit_artifacts = build_implicit_artifacts(train_fit)
val_d_known_mask = val_fit[COL_USER].isin(model_d_fit_artifacts["user_enc"]) & val_fit[COL_ITEM].isin(model_d_fit_artifacts["item_enc"])

print("train_fit implicit shape:", model_d_fit_artifacts["C"].shape)
print("train_fit implicit nnz  :", model_d_fit_artifacts["C"].nnz)
print("alpha_fit              :", model_d_fit_artifacts["alpha"])
print("val_fit known coverage :", round(float(val_d_known_mask.mean()), 6))
print("val_fit date range     :", date_range_str(val_fit))

res_val_base_d = evaluate_predictions(y_val_true_oof, baseline_val_preds_oof, name="baseline_on_val_fit_model_d")
print_eval(res_val_base_d)


MODEL D ? BUILD train_fit ARTIFACTS
train_fit implicit shape: (601727, 166990)
train_fit implicit nnz  : 4027534
alpha_fit              : 1.0
val_fit known coverage : 0.831032
val_fit date range     : 2017-01-12 00:00:00 -> 2017-05-15 00:00:00
Model: baseline_on_val_fit_model_d
N    : 449825
RMSE : 1.094414
MAE  : 0.792196
------------------------------------------------------------
star_1_rmse   : 3.101513385772705
star_1_mae    : 3.0546438694000244
star_1_n      : 24386
star_2_rmse   : 2.181936264038086
star_2_mae    : 2.1269261837005615
star_2_n      : 18751
star_3_rmse   : 1.2853533029556274
star_3_mae    : 1.2179089784622192
star_3_n      : 32659
star_4_rmse   : 0.4962732791900635
star_4_mae    : 0.4226570427417755
star_4_n      : 62459
star_5_rmse   : 0.6835676431655884
star_5_mae    : 0.5642480254173279
star_5_n      : 311570


In [22]:
# ============================================
# CELL 21 ? MODEL D FAST PREDICT HELPERS
# ============================================

def predict_als_rating_fast(baseline_arr, u_idx, i_idx, U, V, residual_scale=0.1, clip_lo=1.0, clip_hi=5.0):
    scores = np.sum(U[u_idx] * V[i_idx], axis=1).astype(np.float32)
    preds = np.clip(baseline_arr + residual_scale * scores, clip_lo, clip_hi).astype(np.float32)
    return preds, scores

def predict_als_with_fallback(df_subset, baseline_arr, implicit_artifacts, U, V, residual_scale):
    preds = baseline_arr.copy().astype(np.float32)
    scores = np.zeros(len(df_subset), dtype=np.float32)
    known_mask = df_subset[COL_USER].isin(implicit_artifacts["user_enc"]) & df_subset[COL_ITEM].isin(implicit_artifacts["item_enc"])
    if known_mask.any():
        known_df = df_subset.loc[known_mask].copy()
        u_idx = known_df[COL_USER].map(implicit_artifacts["user_enc"]).astype(np.int32).values
        i_idx = known_df[COL_ITEM].map(implicit_artifacts["item_enc"]).astype(np.int32).values
        pred_known, score_known = predict_als_rating_fast(baseline_arr[known_mask.to_numpy()], u_idx, i_idx, U, V, residual_scale=residual_scale)
        preds[known_mask.to_numpy()] = pred_known
        scores[known_mask.to_numpy()] = score_known
    return preds.astype(np.float32), scores.astype(np.float32), known_mask

print("Fast predict helper ready.")


Fast predict helper ready.


In [23]:
# ============================================
# CELL 22 ? MODEL D COARSE TUNING ON val_fit
# ============================================

print("=" * 70)
print("MODEL D ? COARSE TUNING ON val_fit")
print("=" * 70)

model_d_fallback_only = False
model_d_error_message = None
results_d_scale_df = pd.DataFrame()
best_rs_d_init = 0.0
U_d_init = None
V_d_init = None
hist_d_init = []
scale_grid_d = [-0.50, -0.25, -0.10, -0.05, 0.00, 0.01, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50, 0.75, 1.00]

try:
    U_d_init, V_d_init, hist_d_init = als_implicit_train(model_d_fit_artifacts["C"].astype(np.float32), n_factors=20, n_iter=10, reg=0.01, alpha=model_d_fit_artifacts["alpha"], random_state=42, verbose=True)
    results_d_scale = []
    for rs in scale_grid_d:
        y_val_pred_tmp, scores_tmp, known_mask_tmp = predict_als_with_fallback(val_fit, baseline_val_preds_oof, model_d_fit_artifacts, U_d_init, V_d_init, rs)
        res_tmp = evaluate_predictions(y_val_true_oof, y_val_pred_tmp, name=f"d_val_rs_{rs}")
        results_d_scale.append({"residual_scale": float(rs), "rmse": float(res_tmp["rmse"]), "mae": float(res_tmp["mae"]), "known_coverage": float(known_mask_tmp.mean()), "score_mean": float(np.mean(scores_tmp)), "score_std": float(np.std(scores_tmp))})
        print(f"rs={rs:+.3f} | VAL RMSE={res_tmp['rmse']:.6f} | VAL MAE={res_tmp['mae']:.6f}")
    results_d_scale_df = pd.DataFrame(results_d_scale).sort_values(["rmse", "mae", "residual_scale"]).reset_index(drop=True)
    best_rs_d_init = float(results_d_scale_df.iloc[0]["residual_scale"])
except Exception as exc:
    model_d_fallback_only = True
    model_d_error_message = repr(exc)
    results_d_scale_df = pd.DataFrame([{"residual_scale": 0.0, "rmse": float(result_val_baseline_oof["rmse"]), "mae": float(result_val_baseline_oof["mae"]), "known_coverage": float(val_d_known_mask.mean()), "score_mean": 0.0, "score_std": 0.0}])
    print("Model D coarse tuning failed. Fallback to baseline OOF predictions.")
    print(model_d_error_message)

display(results_d_scale_df)
print("Best coarse residual_scale:", best_rs_d_init)


MODEL D ? COARSE TUNING ON val_fit


  0%|          | 0/10 [00:00<?, ?it/s]

rs=-0.500 | VAL RMSE=1.094420 | VAL MAE=0.792251
rs=-0.250 | VAL RMSE=1.094417 | VAL MAE=0.792223
rs=-0.100 | VAL RMSE=1.094415 | VAL MAE=0.792207
rs=-0.050 | VAL RMSE=1.094415 | VAL MAE=0.792201
rs=+0.000 | VAL RMSE=1.094414 | VAL MAE=0.792196
rs=+0.010 | VAL RMSE=1.094414 | VAL MAE=0.792195
rs=+0.030 | VAL RMSE=1.094414 | VAL MAE=0.792193
rs=+0.050 | VAL RMSE=1.094414 | VAL MAE=0.792191
rs=+0.080 | VAL RMSE=1.094414 | VAL MAE=0.792188
rs=+0.100 | VAL RMSE=1.094414 | VAL MAE=0.792186
rs=+0.150 | VAL RMSE=1.094413 | VAL MAE=0.792181
rs=+0.200 | VAL RMSE=1.094413 | VAL MAE=0.792175
rs=+0.300 | VAL RMSE=1.094413 | VAL MAE=0.792165
rs=+0.500 | VAL RMSE=1.094413 | VAL MAE=0.792146
rs=+0.750 | VAL RMSE=1.094414 | VAL MAE=0.792122
rs=+1.000 | VAL RMSE=1.094416 | VAL MAE=0.792100


,residual_scale,rmse,mae,known_coverage,score_mean,score_std
0,0.50,1.094413,0.792146,0.831032,0.000247,0.00484
1,0.30,1.094413,0.792165,0.831032,0.000247,0.00484
2,0.20,1.094413,0.792175,0.831032,0.000247,0.00484
3,0.15,1.094413,0.792181,0.831032,0.000247,0.00484
4,0.10,1.094414,0.792186,0.831032,0.000247,0.00484
5,0.08,1.094414,0.792188,0.831032,0.000247,0.00484
6,0.05,1.094414,0.792191,0.831032,0.000247,0.00484
7,0.03,1.094414,0.792193,0.831032,0.000247,0.00484
8,0.01,1.094414,0.792195,0.831032,0.000247,0.00484
9,0.75,1.094414,0.792122,0.831032,0.000247,0.00484


Best coarse residual_scale: 0.5


In [24]:
# ============================================
# CELL 23 ? MODEL D FULL TUNING ON val_fit + SAVE OOF
# ============================================

print("=" * 70)
print("MODEL D ? FULL TUNING ON val_fit")
print("=" * 70)

factor_grid_d = [20, 50]
reg_grid_d = [0.005, 0.01]
iter_grid_d = [10, 15]
results_d_full = []
best_model_d = None
best_model_d_key = (float("inf"), float("inf"))

if not model_d_fallback_only:
    try:
        fine_scale_grid_d = sorted(set(list(np.round(np.linspace(0.8, 1.5, 15), 3)) + [0.0, float(best_rs_d_init)]))
        print("Fine scale grid:", fine_scale_grid_d)
        for n_factors in factor_grid_d:
            for reg in reg_grid_d:
                for n_iter in iter_grid_d:
                    print(f"\nTraining ALS: k={n_factors} | reg={reg} | iter={n_iter}")
                    U_tmp, V_tmp, hist_tmp = als_implicit_train(model_d_fit_artifacts["C"].astype(np.float32), n_factors=n_factors, n_iter=n_iter, reg=reg, alpha=model_d_fit_artifacts["alpha"], random_state=42, verbose=False)
                    best_rs_for_model = None
                    best_rmse_for_model = float("inf")
                    best_mae_for_model = float("inf")
                    best_pred_for_model = None
                    best_scores_for_model = None
                    for rs in fine_scale_grid_d:
                        y_val_pred_tmp, scores_tmp, _ = predict_als_with_fallback(val_fit, baseline_val_preds_oof, model_d_fit_artifacts, U_tmp, V_tmp, rs)
                        res_tmp = evaluate_predictions(y_val_true_oof, y_val_pred_tmp, name=f"d_val_k{n_factors}_reg{reg}_iter{n_iter}_rs{rs}")
                        cur_key = (float(res_tmp["rmse"]), float(res_tmp["mae"]))
                        if cur_key < (best_rmse_for_model, best_mae_for_model):
                            best_rmse_for_model = cur_key[0]
                            best_mae_for_model = cur_key[1]
                            best_rs_for_model = float(rs)
                            best_pred_for_model = y_val_pred_tmp.copy()
                            best_scores_for_model = scores_tmp.copy()
                    results_d_full.append({"n_factors": int(n_factors), "reg": float(reg), "n_iter": int(n_iter), "best_residual_scale": float(best_rs_for_model), "val_rmse": float(best_rmse_for_model), "val_mae": float(best_mae_for_model), "final_approx_loss": float(hist_tmp[-1]["approx_loss"])})
                    print(f"-> best_rs={best_rs_for_model:+.4f} | VAL RMSE={best_rmse_for_model:.6f} | VAL MAE={best_mae_for_model:.6f}")
                    cur_model_key = (float(best_rmse_for_model), float(best_mae_for_model))
                    if cur_model_key < best_model_d_key:
                        best_model_d_key = cur_model_key
                        best_model_d = {"U": U_tmp.copy(), "V": V_tmp.copy(), "history": hist_tmp, "config": {"n_factors": int(n_factors), "reg": float(reg), "n_iter": int(n_iter), "residual_scale": float(best_rs_for_model)}, "val_rmse": float(best_rmse_for_model), "val_mae": float(best_mae_for_model), "val_preds": best_pred_for_model, "val_scores": best_scores_for_model}
    except Exception as exc:
        model_d_fallback_only = True
        model_d_error_message = repr(exc)
        best_model_d = None
        print("Model D full tuning failed. Fallback to baseline OOF predictions.")
        print(model_d_error_message)

results_d_full_df = pd.DataFrame(results_d_full)
if len(results_d_full_df) > 0:
    results_d_full_df = results_d_full_df.sort_values(["val_rmse", "val_mae", "n_factors", "reg", "n_iter"]).reset_index(drop=True)
display(results_d_full_df)

if model_d_fallback_only or best_model_d is None:
    best_cfg_d = {"n_factors": None, "reg": None, "n_iter": None, "residual_scale": 0.0, "fallback_to_baseline": True, "error": model_d_error_message}
    model_d_val_preds_oof = baseline_val_preds_oof.copy()
    model_d_val_scores_oof = np.zeros(len(val_fit), dtype=np.float32)
else:
    best_cfg_d = best_model_d["config"]
    model_d_val_preds_oof = best_model_d["val_preds"].astype(np.float32)
    model_d_val_scores_oof = best_model_d["val_scores"].astype(np.float32)

model_d_val_result_oof = evaluate_predictions(y_val_true_oof, model_d_val_preds_oof, name="model_d_val_oof")
print_eval(model_d_val_result_oof)
np.save(STAGE3_DIR / "model_d_val_preds_oof.npy", model_d_val_preds_oof)
np.save(STAGE3_DIR / "model_d_val_scores_oof.npy", model_d_val_scores_oof)
results_d_full_df.to_csv(STAGE3_DIR / "model_d_tuning.csv", index=False)


MODEL D ? FULL TUNING ON val_fit
Fine scale grid: [0.0, 0.5, np.float64(0.8), np.float64(0.85), np.float64(0.9), np.float64(0.95), np.float64(1.0), np.float64(1.05), np.float64(1.1), np.float64(1.15), np.float64(1.2), np.float64(1.25), np.float64(1.3), np.float64(1.35), np.float64(1.4), np.float64(1.45), np.float64(1.5)]

Training ALS: k=20 | reg=0.005 | iter=10
-> best_rs=+0.5000 | VAL RMSE=1.094413 | VAL MAE=0.792146

Training ALS: k=20 | reg=0.005 | iter=15
-> best_rs=+0.0000 | VAL RMSE=1.094414 | VAL MAE=0.792196

Training ALS: k=20 | reg=0.01 | iter=10
-> best_rs=+0.5000 | VAL RMSE=1.094413 | VAL MAE=0.792146

Training ALS: k=20 | reg=0.01 | iter=15
-> best_rs=+0.0000 | VAL RMSE=1.094414 | VAL MAE=0.792196

Training ALS: k=50 | reg=0.005 | iter=10
-> best_rs=+0.0000 | VAL RMSE=1.094414 | VAL MAE=0.792196

Training ALS: k=50 | reg=0.005 | iter=15
-> best_rs=+0.0000 | VAL RMSE=1.094414 | VAL MAE=0.792196

Training ALS: k=50 | reg=0.01 | iter=10
-> best_rs=+0.0000 | VAL RMSE=1.094414

,n_factors,reg,n_iter,best_residual_scale,val_rmse,val_mae,final_approx_loss
0,20,0.005,10,0.5,1.094413,0.792146,NaN
1,20,0.010,10,0.5,1.094413,0.792146,NaN
2,20,0.005,15,0.0,1.094414,0.792196,NaN
3,20,0.010,15,0.0,1.094414,0.792196,NaN
4,50,0.005,10,0.0,1.094414,0.792196,NaN
5,50,0.005,15,0.0,1.094414,0.792196,NaN
6,50,0.010,10,0.0,1.094414,0.792196,NaN
7,50,0.010,15,0.0,1.094414,0.792196,NaN


Model: model_d_val_oof
N    : 449825
RMSE : 1.094413
MAE  : 0.792146
------------------------------------------------------------
star_1_rmse   : 3.10160231590271
star_1_mae    : 3.054734230041504
star_1_n      : 24386
star_2_rmse   : 2.182044744491577
star_2_mae    : 2.127035617828369
star_2_n      : 18751
star_3_rmse   : 1.2854692935943604
star_3_mae    : 1.218029260635376
star_3_n      : 32659
star_4_rmse   : 0.49637070298194885
star_4_mae    : 0.42276400327682495
star_4_n      : 62459
star_5_rmse   : 0.6834759712219238
star_5_mae    : 0.5641270279884338
star_5_n      : 311570


In [25]:
# ============================================
# CELL 24 ? REFIT BEST MODEL D ON FULL TRAIN
# ============================================

print("=" * 70)
print("MODEL D ? REFIT BEST CONFIG ON FULL TRAIN")
print("=" * 70)
print(best_cfg_d)

model_d_full_artifacts = build_implicit_artifacts(train)
ALPHA_ALS_FULL = float(model_d_full_artifacts["alpha"])
U_d_best = np.zeros((1, 1), dtype=np.float32)
V_d_best = np.zeros((1, 1), dtype=np.float32)
hist_d_best = []
model_d_refit_status = "baseline_fallback"

if not bool(best_cfg_d.get("fallback_to_baseline", False)):
    try:
        U_d_best, V_d_best, hist_d_best = als_implicit_train(model_d_full_artifacts["C"].astype(np.float32), n_factors=best_cfg_d["n_factors"], n_iter=best_cfg_d["n_iter"], reg=best_cfg_d["reg"], alpha=ALPHA_ALS_FULL, random_state=42, verbose=True)
        model_d_refit_status = "trained"
    except Exception as exc:
        model_d_refit_status = "baseline_fallback"
        model_d_error_message = repr(exc)
        best_cfg_d["fallback_to_baseline"] = True
        best_cfg_d["full_refit_error"] = model_d_error_message
        print("Model D full refit failed. Falling back to baseline on test_warm.")
        print(model_d_error_message)


MODEL D ? REFIT BEST CONFIG ON FULL TRAIN
{'n_factors': 20, 'reg': 0.005, 'n_iter': 10, 'residual_scale': 0.5}


  0%|          | 0/10 [00:00<?, ?it/s]

In [26]:
# ============================================
# CELL 25 ? MODEL D FINAL TEST EVALUATION
# ============================================

print("=" * 70)
print("MODEL D ? FINAL TEST EVALUATION")
print("=" * 70)

if bool(best_cfg_d.get("fallback_to_baseline", False)):
    model_d_preds = y_pred_base.copy()
    scores_d_best = np.zeros(len(test_warm), dtype=np.float32)
    known_mask_test_d = np.zeros(len(test_warm), dtype=bool)
else:
    model_d_preds, scores_d_best, known_mask_test_d = predict_als_with_fallback(test_warm, y_pred_base, model_d_full_artifacts, U_d_best, V_d_best, float(best_cfg_d["residual_scale"]))

assert len(model_d_preds) == len(test_warm), "Model D final predictions must align with test_warm."
result_d_best = evaluate_predictions(y_true_warm, model_d_preds, name="model_d_als_full_train")
result_base_d_final = evaluate_predictions(y_true_warm, y_pred_base, name="baseline_on_test_warm")
print_eval(result_base_d_final)
print_eval(result_d_best)
rmse_gain_d = result_base_d_final["rmse"] - result_d_best["rmse"]
mae_gain_d = result_base_d_final["mae"] - result_d_best["mae"]
print("RMSE gain over baseline test_warm:", rmse_gain_d)
print("MAE  gain over baseline test_warm:", mae_gain_d)
print("\nALS score distribution:")
print(pd.Series(scores_d_best).describe())


MODEL D ? FINAL TEST EVALUATION
Model: baseline_on_test_warm
N    : 935578
RMSE : 1.144186
MAE  : 0.855168
------------------------------------------------------------
star_1_rmse   : 3.215768337249756
star_1_mae    : 3.2064449787139893
star_1_n      : 60306
star_2_rmse   : 2.2479584217071533
star_2_mae    : 2.236931562423706
star_2_n      : 44048
star_3_rmse   : 1.2934361696243286
star_3_mae    : 1.2766953706741333
star_3_n      : 70544
star_4_rmse   : 0.3738032281398773
star_4_mae    : 0.3362816274166107
star_4_n      : 126808
star_5_rmse   : 0.6189844608306885
star_5_mae    : 0.5923428535461426
star_5_n      : 633872
Model: model_d_als_full_train
N    : 935578
RMSE : 1.144198
MAE  : 0.855108
------------------------------------------------------------
star_1_rmse   : 3.215909242630005
star_1_mae    : 3.2065839767456055
star_1_n      : 60306
star_2_rmse   : 2.2481167316436768
star_2_mae    : 2.237086296081543
star_2_n      : 44048
star_3_rmse   : 1.2936094999313354
star_3_mae    : 1.

In [27]:
# ============================================
# CELL 26 ? MODEL D SAVE
# ============================================

print("=" * 70)
print("SAVING MODEL D ARTIFACTS")
print("=" * 70)

model_d_summary = {"best_config": to_jsonable(best_cfg_d), "alpha_fit": float(model_d_fit_artifacts["alpha"]), "alpha_full": float(ALPHA_ALS_FULL), "val_fit_rmse": float(model_d_val_result_oof["rmse"]), "val_fit_mae": float(model_d_val_result_oof["mae"]), "val_fit_known_coverage": float(val_d_known_mask.mean()), "test_warm_rmse": float(result_d_best["rmse"]), "test_warm_mae": float(result_d_best["mae"]), "rmse_gain_vs_baseline_test_warm": float(rmse_gain_d), "mae_gain_vs_baseline_test_warm": float(mae_gain_d), "test_known_coverage": float(known_mask_test_d.mean()), "per_star": result_d_best["per_star"], "train_history": hist_d_best, "refit_status": model_d_refit_status, "fallback_to_baseline": bool(best_cfg_d.get("fallback_to_baseline", False)), "error": model_d_error_message, "test_warm_used_only_for_final_evaluation": True}

np.save(STAGE3_DIR / "model_d_preds.npy", model_d_preds)
np.save(STAGE3_DIR / "model_d_scores.npy", scores_d_best)
np.save(STAGE3_DIR / "model_d_U.npy", U_d_best)
np.save(STAGE3_DIR / "model_d_V.npy", V_d_best)
with open(STAGE3_DIR / "model_d_result.json", "w") as f:
    json.dump(to_jsonable(result_d_best), f, indent=2)
with open(STAGE3_DIR / "model_d_summary.json", "w") as f:
    json.dump(to_jsonable(model_d_summary), f, indent=2)


SAVING MODEL D ARTIFACTS


In [28]:
# ============================================
# CELL 27 ? FINAL LENGTH CHECKS
# ============================================

final_test_pred_arrays = {"baseline_test": y_pred_base, "model_a_preds": model_a_preds, "model_b_preds": model_b_preds, "model_c_preds": model_c_preds, "model_d_preds": model_d_preds}
for arr_name, arr in final_test_pred_arrays.items():
    assert len(arr) == len(test_warm), f"Length mismatch for {arr_name}: {len(arr)} vs {len(test_warm)}"
    assert np.isfinite(arr).all(), f"Non-finite values found in {arr_name}"

print("All final test_warm prediction lengths are aligned:", len(test_warm))


All final test_warm prediction lengths are aligned: 935578


In [29]:
# ============================================
# CELL 28 - EXPORT STRICT OOF VALIDATION PREDICTIONS FOR STAGE 4
# ============================================

print('=' * 70)
print('EXPORT STRICT OOF VALIDATION PREDICTIONS FOR STAGE 4')
print('=' * 70)

# Gom các mảng dự đoán OOF (Đảm bảo các biến model_x_val_preds_oof đã tồn tại từ các cell trước)
oof_pred_arrays = {
    "y_val_true_oof": y_val_true_oof, 
    "baseline_val_preds_oof": baseline_val_preds_oof, 
    "model_a_val_preds_oof": model_a_val_preds_oof, 
    "model_b_val_preds_oof": model_b_val_preds_oof, 
    "model_c_val_preds_oof": model_c_val_preds_oof, 
    "model_d_val_preds_oof": model_d_val_preds_oof
}

for arr_name, arr in oof_pred_arrays.items():
    assert len(arr) == len(y_val_true_oof), f"Length mismatch for {arr_name}: {len(arr)} vs {len(y_val_true_oof)}"
    assert np.isfinite(arr).all(), f"Non-finite values found in {arr_name}"

# Lưu các file binary và index
validation_index_oof_df.to_csv(STAGE3_DIR / 'validation_index_oof.csv', index=False)
np.save(STAGE3_DIR / 'y_val_true_oof.npy', y_val_true_oof)
np.save(STAGE3_DIR / 'baseline_val_preds_oof.npy', baseline_val_preds_oof)
np.save(STAGE3_DIR / 'model_a_val_preds_oof.npy', model_a_val_preds_oof)
np.save(STAGE3_DIR / 'model_b_val_preds_oof.npy', model_b_val_preds_oof)
np.save(STAGE3_DIR / 'model_c_val_preds_oof.npy', model_c_val_preds_oof)
np.save(STAGE3_DIR / 'model_d_val_preds_oof.npy', model_d_val_preds_oof)

# Cập nhật thông tin cấu hình (Sửa phần model_c để khớp với best_wb, best_wc, best_th)
validation_preds_summary_oof = {
    'strict_oof': True, 
    'split_strategy': 'train sorted by review_date; train_fit = first 90%; val_fit = last 10%', 
    'n_train': int(len(train)), 
    'n_train_fit': int(len(train_fit)), 
    'n_val_fit': int(len(val_fit)), 
    'date_ranges': {
        'train_fit': date_range_str(train_fit), 
        'val_fit': date_range_str(val_fit), 
        'test_warm': date_range_str(test_warm)
    }, 
    'best_configs': {
        'model_a': {
            'topk': int(best_topk), 
            'min_common_users': int(best_mc), 
            'min_neighbors_for_pred': int(best_mn), 
            'shrinkage': float(best_sh), 
            'min_sim_threshold': float(best_ms)
        }, 
        'model_b': to_jsonable(best_cfg_b), 
        # --- PHẦN ĐÃ SỬA CHO MODEL C ---
        'model_c': {
            'best_w_base': float(best_wb), 
            'best_w_content': float(best_wc), 
            'best_sim_threshold': float(best_th)
        }, 
        # ------------------------------
        'model_d': to_jsonable(best_cfg_d)
    }, 
    'metrics_on_val_fit': {
        'baseline': result_val_baseline_oof, 
        'model_a': model_a_val_result_oof, 
        'model_b': model_b_val_result_oof, 
        'model_c': model_c_val_result_oof, 
        'model_d': model_d_val_result_oof
    }, 
    'test_warm_policy': 'test_warm is used only for final evaluation, not tuning'
}

with open(STAGE3_DIR / 'validation_preds_summary_oof.json', 'w') as f:
    json.dump(to_jsonable(validation_preds_summary_oof), f, indent=2)

print(f"Summary saved to {STAGE3_DIR / 'validation_preds_summary_oof.json'}")

EXPORT STRICT OOF VALIDATION PREDICTIONS FOR STAGE 4
Summary saved to /kaggle/working/data/stage3_artifacts/validation_preds_summary_oof.json
